# Read this before running the notebook

**Historical experiment notebook — do not use “Run All.”** It contains setup and generation cells that call the Groq API, including resumable LLM-evaluation cells. The 240-row primary CSV is frozen; do not rerun those cells merely to reproduce the analysis.

**Important design-version note:** earlier pilot cells in this notebook use a fixed-size context-replacement design. The frozen 240-row primary experiment is a later appended-context design: it retrieves the original top five first, then appends distractor chunks. Do not mix the pilot design or its outputs with the primary result table.

The current primary experiment evaluates answer quality and generation-call latency, not retrieval-ranking robustness. Top-five preservation and zero intrusion are construction diagnostics only.

For deterministic statistical recomputation without LLM/API calls, use `scripts/recompute_primary_statistics.py`. A clean end-to-end notebook execution has not yet been certified.

In [1]:
import os

print("Kaggle input files:\n")

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for file in files:
        print(f"{indent}  {file}")

Kaggle input files:

input/
  competitions/
    agent-eval-part-i-grounded-rag-benchmark/
      embedding_model_info.txt
      test_question_embeddings.npy
      corpus.csv
      sample_submission.csv
      corpus_chunk_ids.npy
      agent_eval_dataset.csv
      corpus_embeddings.npy
      documents.csv
      questions.csv
      test_question_ids.npy
      train.csv
      metaData.csv
      test.csv
      solution.csv


In [2]:
import pandas as pd
import os

BASE = "/kaggle/input/competitions/agent-eval-part-i-grounded-rag-benchmark"

corpus = pd.read_csv(os.path.join(BASE, "corpus.csv"))
train = pd.read_csv(os.path.join(BASE, "train.csv"))
test = pd.read_csv(os.path.join(BASE, "test.csv"))

print("CORPUS")
print("Shape:", corpus.shape)
print("Columns:", corpus.columns.tolist())
print()

print("TRAIN")
print("Shape:", train.shape)
print("Columns:", train.columns.tolist())
print()

print("TEST")
print("Shape:", test.shape)
print("Columns:", test.columns.tolist())

CORPUS
Shape: (3600, 5)
Columns: ['chunk_id', 'doc_id', 'title', 'doc_type', 'chunk_text']

TRAIN
Shape: (2200, 4)
Columns: ['question_id', 'question', 'answer', 'gold_chunk_ids']

TEST
Shape: (800, 2)
Columns: ['question_id', 'question']


In [3]:
print("=== CORPUS SAMPLE ===")
display(corpus.head(3))

print("\n=== TRAIN SAMPLE ===")
display(train.head(3))

print("\n=== TEST SAMPLE ===")
display(test.head(3))

=== CORPUS SAMPLE ===


,chunk_id,doc_id,title,doc_type,chunk_text
0,doc_0001_chunk_1,doc_0001,Architecture note: CDN / Data Pipeline / Playb...,architecture_note,The streaming platform consists of core servic...
1,doc_0001_chunk_2,doc_0001,Architecture note: CDN / Data Pipeline / Playb...,architecture_note,Authentication tokens are validated before pla...
2,doc_0001_chunk_3,doc_0001,Architecture note: CDN / Data Pipeline / Playb...,architecture_note,"Observability combines metrics, traces, and lo..."



=== TRAIN SAMPLE ===


,question_id,question,answer,gold_chunk_ids
0,q_0001,"Based on the knowledge base, summarize the key...",Authentication tokens are validated before pla...,['doc_0154_chunk_2']
1,q_0002,Answer using evidence. What is the correct key...,"On 2026-06-07, the data_pipeline_break impacte...",['doc_1110_chunk_1']
2,q_0003,Combine evidence from 'EU cdn_issue incident r...,"On 2025-07-13, the cdn_issue impacted the EU r...","['doc_0733_chunk_1', 'doc_1087_chunk_3']"



=== TEST SAMPLE ===


,question_id,question
0,q_2201,Combine evidence from 'APAC payment_failure in...
1,q_2202,"Based on the knowledge base, summarize the key..."
2,q_2203,Answer using evidence. What is the correct key...


In [4]:
print("=== CORPUS INFO ===")
display(corpus.info())

print("\n=== TRAIN MISSING VALUES ===")
display(train.isna().sum())

print("\n=== TEST MISSING VALUES ===")
display(test.isna().sum())

=== CORPUS INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3600 entries, 0 to 3599
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   chunk_id    3600 non-null   object
 1   doc_id      3600 non-null   object
 2   title       3600 non-null   object
 3   doc_type    3600 non-null   object
 4   chunk_text  3600 non-null   object
dtypes: object(5)
memory usage: 140.8+ KB


None


=== TRAIN MISSING VALUES ===


question_id       0
question          0
answer            0
gold_chunk_ids    0
dtype: int64


=== TEST MISSING VALUES ===


question_id    0
question       0
dtype: int64

In [5]:
import numpy as np

print("=== EMBEDDING MODEL INFO ===")
with open(os.path.join(BASE, "embedding_model_info.txt"), "r") as f:
    print(f.read())

print("\n=== CORPUS EMBEDDINGS ===")
corpus_embeddings = np.load(os.path.join(BASE, "corpus_embeddings.npy"))
print("Shape:", corpus_embeddings.shape)
print("Dtype:", corpus_embeddings.dtype)

print("\n=== TEST QUESTION EMBEDDINGS ===")
test_embeddings = np.load(os.path.join(BASE, "test_question_embeddings.npy"))
print("Shape:", test_embeddings.shape)
print("Dtype:", test_embeddings.dtype)

print("\n=== CHUNK IDS ===")
corpus_chunk_ids = np.load(
    os.path.join(BASE, "corpus_chunk_ids.npy"),
    allow_pickle=True
)
print("Shape:", corpus_chunk_ids.shape)
print("First 5:", corpus_chunk_ids[:5])

=== EMBEDDING MODEL INFO ===
AgentEval Part I — Embedding Artifact Info
{
  "model_name": "sentence-transformers/all-MiniLM-L6-v2",
  "embedding_dim": 384,
  "normalized": true,
  "batch_size": 64,
  "num_chunks": 3600,
  "files": {
    "embeddings": "corpus_embeddings.npy",
    "chunk_ids": "corpus_chunk_ids.npy"
  },
  "note": "Embeddings generated offline. Kaggle baseline uses these files with Internet OFF."
}

=== CORPUS EMBEDDINGS ===
Shape: (3600, 384)
Dtype: float32

=== TEST QUESTION EMBEDDINGS ===
Shape: (800, 384)
Dtype: float32

=== CHUNK IDS ===
Shape: (3600,)
First 5: ['doc_0001_chunk_1' 'doc_0001_chunk_2' 'doc_0001_chunk_3'
 'doc_0002_chunk_1' 'doc_0002_chunk_2']


In [6]:
!pip install rank-bm25 faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 68.4 MB/s eta 0:00:00


In [7]:
from rank_bm25 import BM25Okapi
import numpy as np
import pandas as pd
import ast
import faiss
import time

documents = corpus["chunk_text"].tolist()
chunk_ids = corpus["chunk_id"].tolist()

tokenized_documents = [
    doc.lower().split()
    for doc in documents
]

bm25 = BM25Okapi(tokenized_documents)

print("BM25 index built.")
print("Documents:", len(documents))

BM25 index built.
Documents: 3600


In [8]:
# Build dense retrieval index from the provided embeddings

dense_index = faiss.IndexFlatIP(corpus_embeddings.shape[1])

dense_index.add(
    corpus_embeddings.astype("float32")
)

print("Dense index built.")
print("Vectors:", dense_index.ntotal)
print("Dimensions:", dense_index.d)

Dense index built.
Vectors: 3600
Dimensions: 384


In [9]:
def retrieve_bm25(query, k=5):
    scores = bm25.get_scores(query.lower().split())
    top_indices = np.argsort(scores)[::-1][:k]

    return [
        {
            "chunk_id": chunk_ids[i],
            "score": float(scores[i]),
            "index": int(i)
        }
        for i in top_indices
    ]


def retrieve_dense(query_embedding, k=5):
    query_embedding = np.asarray(
        query_embedding,
        dtype="float32"
    ).reshape(1, -1)

    scores, indices = dense_index.search(query_embedding, k)

    return [
        {
            "chunk_id": chunk_ids[i],
            "score": float(scores[0][rank]),
            "index": int(i)
        }
        for rank, i in enumerate(indices[0])
    ]


print("Retrieval functions ready.")

Retrieval functions ready.


In [10]:
# Test retrieval on the first training question

sample = train.iloc[0]

question = sample["question"]
gold_ids = ast.literal_eval(sample["gold_chunk_ids"])

print("QUESTION:")
print(question)

print("\nGOLD CHUNK(S):")
print(gold_ids)

print("\nBM25 TOP 5:")
bm25_results = retrieve_bm25(question, k=5)

for rank, result in enumerate(bm25_results, 1):
    marker = " <-- GOLD" if result["chunk_id"] in gold_ids else ""
    print(
        f"{rank}. {result['chunk_id']} "
        f"(score={result['score']:.4f}){marker}"
    )

print("\nBM25 RETRIEVED TEXT:")
for result in bm25_results[:3]:
    print(f"\n[{result['chunk_id']}]")
    print(corpus.iloc[result["index"]]["chunk_text"])

QUESTION:
Based on the knowledge base, summarize the key point from: Architecture note: Playback API / Observability / Search / Billing.

GOLD CHUNK(S):
['doc_0154_chunk_2']

BM25 TOP 5:
1. doc_0154_chunk_1 (score=10.1203)
2. doc_0332_chunk_1 (score=10.1203)
3. doc_0702_chunk_1 (score=10.1203)
4. doc_0376_chunk_1 (score=10.1203)
5. doc_0833_chunk_1 (score=10.1203)

BM25 RETRIEVED TEXT:

[doc_0154_chunk_1]
The streaming platform consists of core services including Playback API, Observability, Search, Billing. Requests flow from edge caches to regional clusters with automated health checks.

[doc_0332_chunk_1]
The streaming platform consists of core services including Recommendations, Playback API, CDN, Billing. Requests flow from edge caches to regional clusters with automated health checks.

[doc_0702_chunk_1]
The streaming platform consists of core services including Playback API, Observability, Recommendations, Billing. Requests flow from edge caches to regional clusters with automat

In [11]:
!pip install -q sentence-transformers

In [12]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Model loaded.")
print("Embedding dimension:", model.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded.
Embedding dimension: 384


/tmp/ipykernel_17/3484025840.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


In [13]:
train_questions = train["question"].tolist()

train_question_embeddings = model.encode(
    train_questions,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

train_question_embeddings = np.asarray(
    train_question_embeddings,
    dtype="float32"
)

print("Training question embeddings:", train_question_embeddings.shape)

Batches:   0%|          | 0/35 [00:00<?, ?it/s]

Training question embeddings: (2200, 384)


In [14]:
# =========================================================
# REBUILD DENSE FAISS INDEX
# =========================================================

import faiss
import numpy as np

print("Building dense retrieval index...")

dense_index = faiss.IndexFlatIP(
    corpus_embeddings.shape[1]
)

dense_index.add(
    corpus_embeddings.astype("float32")
)

print("=" * 70)
print("DENSE INDEX READY")
print("=" * 70)
print(f"Vectors:    {dense_index.ntotal}")
print(f"Dimensions: {dense_index.d}")

Building dense retrieval index...
DENSE INDEX READY
Vectors:    3600
Dimensions: 384


In [15]:
# Evaluate BM25 and Dense retrieval on all 2,200 training questions

K_VALUES = [1, 3, 5, 10]


def get_gold_ids(value):
    if isinstance(value, list):
        return value
    return ast.literal_eval(value)


def recall_at_k(retrieved_ids, gold_ids, k):
    retrieved = set(retrieved_ids[:k])
    gold = set(gold_ids)

    return len(retrieved & gold) / len(gold)


bm25_recalls = {k: [] for k in K_VALUES}
dense_recalls = {k: [] for k in K_VALUES}

print("Evaluating 2,200 questions...")

for i in range(len(train)):

    question = train.iloc[i]["question"]
    gold_ids = get_gold_ids(train.iloc[i]["gold_chunk_ids"])

    # BM25
    bm25_results = retrieve_bm25(question, k=max(K_VALUES))
    bm25_ids = [r["chunk_id"] for r in bm25_results]

    # Dense
    dense_results = retrieve_dense(
        train_question_embeddings[i],
        k=max(K_VALUES)
    )
    dense_ids = [r["chunk_id"] for r in dense_results]

    # Recall
    for k in K_VALUES:
        bm25_recalls[k].append(
            recall_at_k(bm25_ids, gold_ids, k)
        )

        dense_recalls[k].append(
            recall_at_k(dense_ids, gold_ids, k)
        )


print("\n=== RETRIEVAL RECALL ===")

for k in K_VALUES:
    print(
        f"Recall@{k}: "
        f"BM25 = {np.mean(bm25_recalls[k]):.4f} | "
        f"Dense = {np.mean(dense_recalls[k]):.4f}"
    )

Evaluating 2,200 questions...

=== RETRIEVAL RECALL ===
Recall@1: BM25 = 0.0155 | Dense = 0.0330
Recall@3: BM25 = 0.0391 | Dense = 0.0800
Recall@5: BM25 = 0.0559 | Dense = 0.1009
Recall@10: BM25 = 0.0830 | Dense = 0.1193


In [16]:
def min_max_normalize(scores):
    scores = np.asarray(scores, dtype=float)

    min_score = scores.min()
    max_score = scores.max()

    if max_score == min_score:
        return np.zeros_like(scores)

    return (scores - min_score) / (max_score - min_score)


def retrieve_hybrid(query, query_embedding, k=10, alpha=0.5):

    # BM25 scores aligned with corpus indices
    bm25_scores = np.asarray(
        bm25.get_scores(query.lower().split()),
        dtype=float
    )

    # Calculate dense similarity for every corpus chunk.
    # Because embeddings are normalized, dot product = cosine similarity.
    query_embedding = np.asarray(
        query_embedding,
        dtype="float32"
    ).reshape(1, -1)

    dense_scores = np.dot(
        corpus_embeddings,
        query_embedding.T
    ).flatten()

    # Normalize both score distributions
    bm25_norm = min_max_normalize(bm25_scores)
    dense_norm = min_max_normalize(dense_scores)

    # Combine scores
    hybrid_scores = (
        (1 - alpha) * bm25_norm
        + alpha * dense_norm
    )

    # Rank corpus indices by combined score
    top_indices = np.argsort(hybrid_scores)[::-1][:k]

    return [
        {
            "chunk_id": chunk_ids[i],
            "score": float(hybrid_scores[i]),
            "index": int(i)
        }
        for i in top_indices
    ]


print("Corrected hybrid retrieval function ready.")

Corrected hybrid retrieval function ready.


In [17]:
# =========================================================
# HYBRID RETRIEVAL RECALL
# Evaluate Hybrid retrieval on all 2,200 questions
# =========================================================

hybrid_recalls = {
    k: [] for k in K_VALUES
}

print("Evaluating Hybrid retrieval on 2,200 questions...")

for i in range(len(train)):

    question = train.iloc[i]["question"]

    gold_ids = get_gold_ids(
        train.iloc[i]["gold_chunk_ids"]
    )

    hybrid_results = retrieve_hybrid(
        question,
        train_question_embeddings[i],
        k=max(K_VALUES),
        alpha=0.5
    )

    hybrid_ids = [
        r["chunk_id"]
        for r in hybrid_results
    ]

    for k in K_VALUES:

        hybrid_recalls[k].append(
            recall_at_k(
                hybrid_ids,
                gold_ids,
                k
            )
        )


print("\n=== RETRIEVAL RECALL ===")

for k in K_VALUES:

    print(
        f"Recall@{k}: "
        f"BM25 = {np.mean(bm25_recalls[k]):.4f} | "
        f"Dense = {np.mean(dense_recalls[k]):.4f} | "
        f"Hybrid = {np.mean(hybrid_recalls[k]):.4f}"
    )

Evaluating Hybrid retrieval on 2,200 questions...

=== RETRIEVAL RECALL ===
Recall@1: BM25 = 0.0155 | Dense = 0.0330 | Hybrid = 0.0493
Recall@3: BM25 = 0.0391 | Dense = 0.0800 | Hybrid = 0.0925
Recall@5: BM25 = 0.0559 | Dense = 0.1009 | Hybrid = 0.1105
Recall@10: BM25 = 0.0830 | Dense = 0.1193 | Hybrid = 0.1202


In [18]:
hybrid_recalls = {k: [] for k in K_VALUES}

print("Evaluating Hybrid retrieval on 2,200 questions...")

for i in range(len(train)):

    question = train.iloc[i]["question"]
    gold_ids = get_gold_ids(train.iloc[i]["gold_chunk_ids"])

    hybrid_results = retrieve_hybrid(
        question,
        train_question_embeddings[i],
        k=max(K_VALUES),
        alpha=0.5
    )

    hybrid_ids = [r["chunk_id"] for r in hybrid_results]

    for k in K_VALUES:
        hybrid_recalls[k].append(
            recall_at_k(hybrid_ids, gold_ids, k)
        )


print("\n=== RETRIEVAL RECALL ===")

for k in K_VALUES:
    print(
        f"Recall@{k}: "
        f"BM25 = {np.mean(bm25_recalls[k]):.4f} | "
        f"Dense = {np.mean(dense_recalls[k]):.4f} | "
        f"Hybrid = {np.mean(hybrid_recalls[k]):.4f}"
    )

Evaluating Hybrid retrieval on 2,200 questions...

=== RETRIEVAL RECALL ===
Recall@1: BM25 = 0.0155 | Dense = 0.0330 | Hybrid = 0.0493
Recall@3: BM25 = 0.0391 | Dense = 0.0800 | Hybrid = 0.0925
Recall@5: BM25 = 0.0559 | Dense = 0.1009 | Hybrid = 0.1105
Recall@10: BM25 = 0.0830 | Dense = 0.1193 | Hybrid = 0.1202


In [19]:
ALPHAS = [0.0, 0.25, 0.5, 0.75, 1.0]

alpha_results = []

print("Testing hybrid weights...\n")

for alpha in ALPHAS:

    recalls = {k: [] for k in K_VALUES}

    for i in range(len(train)):

        question = train.iloc[i]["question"]
        gold_ids = get_gold_ids(train.iloc[i]["gold_chunk_ids"])

        results = retrieve_hybrid(
            question,
            train_question_embeddings[i],
            k=max(K_VALUES),
            alpha=alpha
        )

        retrieved_ids = [r["chunk_id"] for r in results]

        for k in K_VALUES:
            recalls[k].append(
                recall_at_k(retrieved_ids, gold_ids, k)
            )

    row = {
        "alpha": alpha
    }

    for k in K_VALUES:
        row[f"Recall@{k}"] = np.mean(recalls[k])

    alpha_results.append(row)

alpha_df = pd.DataFrame(alpha_results)

display(alpha_df)

Testing hybrid weights...



,alpha,Recall@1,Recall@3,Recall@5,Recall@10
0,0.00,0.015455,0.039091,0.055909,0.082955
1,0.25,0.047500,0.090455,0.107727,0.117727
2,0.50,0.049318,0.092500,0.110455,0.120227
3,0.75,0.048409,0.092045,0.112273,0.125682
4,1.00,0.032045,0.080455,0.100682,0.116591


In [20]:
# Diagnose why Hybrid alpha=1 differs from Dense retrieval

i = 0

question = train.iloc[i]["question"]
embedding = train_question_embeddings[i]
gold_ids = get_gold_ids(train.iloc[i]["gold_chunk_ids"])

dense_results = retrieve_dense(embedding, k=10)
hybrid_results = retrieve_hybrid(
    question,
    embedding,
    k=10,
    alpha=1.0
)

print("GOLD:")
print(gold_ids)

print("\nDENSE TOP 10:")
for r in dense_results:
    print(r["chunk_id"], r["score"])

print("\nHYBRID alpha=1 TOP 10:")
for r in hybrid_results:
    print(r["chunk_id"], r["score"])

print("\nSame ranking?")
print(
    [r["chunk_id"] for r in dense_results]
    ==
    [r["chunk_id"] for r in hybrid_results]
)

GOLD:
['doc_0154_chunk_2']

DENSE TOP 10:
doc_0154_chunk_1 0.6403231620788574
doc_0763_chunk_1 0.6378600597381592
doc_0290_chunk_1 0.6360288858413696
doc_0807_chunk_1 0.6346941590309143
doc_0702_chunk_1 0.6346286535263062
doc_0957_chunk_1 0.6266261339187622
doc_0663_chunk_1 0.624068021774292
doc_1089_chunk_1 0.6238925457000732
doc_0422_chunk_1 0.622230052947998
doc_0200_chunk_1 0.6206870079040527

HYBRID alpha=1 TOP 10:
doc_0154_chunk_1 1.0
doc_0763_chunk_1 0.9960571460030515
doc_0290_chunk_1 0.9931258624754585
doc_0807_chunk_1 0.9909891798598438
doc_0702_chunk_1 0.9908842253685882
doc_0957_chunk_1 0.9780742434713298
doc_0663_chunk_1 0.9739793008752345
doc_1089_chunk_1 0.9736984999045392
doc_0422_chunk_1 0.9710372356589945
doc_0200_chunk_1 0.9685670840007087

Same ranking?
True


In [21]:
i = 0

query_embedding = train_question_embeddings[i].reshape(1, -1).astype("float32")

# FAISS scores
faiss_scores, faiss_indices = dense_index.search(
    query_embedding,
    len(documents)
)

# Direct dot-product scores
direct_scores = np.dot(
    corpus_embeddings,
    query_embedding.T
).flatten()

faiss_order = faiss_indices[0]
direct_order = np.argsort(direct_scores)[::-1]

print("FAISS top 10:")
print([chunk_ids[i] for i in faiss_order[:10]])

print("\nDirect dot-product top 10:")
print([chunk_ids[i] for i in direct_order[:10]])

print("\nTop-10 ranking identical:")
print(
    np.array_equal(
        faiss_order[:10],
        direct_order[:10]
    )
)

print("\nMaximum score difference:")
print(
    np.max(
        np.abs(
            faiss_scores[0] -
            direct_scores[faiss_order]
        )
    )
)

FAISS top 10:
['doc_0154_chunk_1', 'doc_0763_chunk_1', 'doc_0290_chunk_1', 'doc_0807_chunk_1', 'doc_0702_chunk_1', 'doc_0957_chunk_1', 'doc_0663_chunk_1', 'doc_1089_chunk_1', 'doc_0422_chunk_1', 'doc_0200_chunk_1']

Direct dot-product top 10:
['doc_0154_chunk_1', 'doc_0763_chunk_1', 'doc_0290_chunk_1', 'doc_0807_chunk_1', 'doc_0702_chunk_1', 'doc_0957_chunk_1', 'doc_0663_chunk_1', 'doc_1089_chunk_1', 'doc_0422_chunk_1', 'doc_0200_chunk_1']

Top-10 ranking identical:
True

Maximum score difference:
1.1920929e-07


In [22]:
sample_rows = train.sample(5, random_state=42)

for _, row in sample_rows.iterrows():
    gold_ids = get_gold_ids(row["gold_chunk_ids"])

    print("=" * 80)
    print("Question:")
    print(row["question"])

    print("\nGold chunks:")
    print(gold_ids)

    for chunk_id in gold_ids:
        chunk = corpus[
            corpus["chunk_id"] == chunk_id
        ]["chunk_text"].iloc[0]

        print(f"\n[{chunk_id}]")
        print(chunk)

Question:
Combine evidence from 'EU playback_error incident report (2026-03-26)' and 'Deployment Log: operational notes'. What is the recommended mitigation approach?

Gold chunks:
['doc_0730_chunk_2', 'doc_0007_chunk_3']

[doc_0730_chunk_2]
Initial triage reviewed dashboards and logs to reduce uncertainty. Root cause analysis identified dependency_timeout as the primary driver of the incident.

[doc_0007_chunk_3]
Use canary checks after applying changes, and prefer reversible actions under high uncertainty. Document outcomes and share a short incident summary for stakeholder visibility.
Question:
According to the SLA policy, what is the acknowledge and mitigation target for Severity 4 incidents?

Gold chunks:
['doc_0496_chunk_1']

[doc_0496_chunk_1]
This policy defines response targets by severity. Severity 5 (global outage): acknowledge within 5 minutes, mitigation within 30 minutes.
Question:
Answer using evidence. What is the correct key detail from 'EU payment_failure incident rep

In [23]:
for _, row in sample_rows.iterrows():
    print("=" * 80)
    print("Question:")
    print(row["question"])

    print("\nReference answer:")
    print(row["answer"])

    print("\nGold chunks:")
    print(get_gold_ids(row["gold_chunk_ids"]))

Question:
Combine evidence from 'EU playback_error incident report (2026-03-26)' and 'Deployment Log: operational notes'. What is the recommended mitigation approach?

Reference answer:
Initial triage reviewed dashboards and logs to reduce uncertainty. Use canary checks after applying changes, and prefer reversible actions under high uncertainty.

Gold chunks:
['doc_0730_chunk_2', 'doc_0007_chunk_3']
Question:
According to the SLA policy, what is the acknowledge and mitigation target for Severity 4 incidents?

Reference answer:
Acknowledge within 10 minutes, mitigation within 60 minutes.

Gold chunks:
['doc_0496_chunk_1']
Question:
Answer using evidence. What is the correct key detail from 'EU payment_failure incident report (2026-06-20)' (ignore unrelated notes)?

Reference answer:
Initial triage reviewed dashboards and logs to reduce uncertainty

Gold chunks:
['doc_0438_chunk_2']
Question:
Based on the knowledge base, summarize the key point from: APAC auth_failure incident report (2

In [24]:
# ============================================================
# CONTROLLED NOISE PILOT
# No LLM/API calls yet
# ============================================================

import random
import pandas as pd

PILOT_SIZE = 20
TOP_K = 5
NOISE_LEVELS = [0.00, 0.25, 0.50, 0.75]

# Keep original training indices so embeddings map correctly
pilot = train.sample(
    PILOT_SIZE,
    random_state=42
).copy()

def get_random_noise_chunks(exclude_ids, n, seed):
    available = [
        cid for cid in chunk_ids
        if cid not in set(exclude_ids)
    ]

    rng = random.Random(seed)
    return rng.sample(available, n)


def build_noisy_context(retrieved_ids, noise_level, seed):
    """
    Keep total context size fixed at TOP_K.

    Increasing noise replaces some retrieved chunks
    with randomly selected irrelevant chunks.
    """

    total = len(retrieved_ids)

    n_noise = int(round(total * noise_level))
    n_relevant = total - n_noise

    # Keep highest-ranked retrieved chunks
    relevant_ids = retrieved_ids[:n_relevant]

    # Select unrelated corpus chunks
    noise_ids = get_random_noise_chunks(
        exclude_ids=retrieved_ids,
        n=n_noise,
        seed=seed
    )

    context_ids = relevant_ids + noise_ids

    # Randomize context order
    rng = random.Random(seed)
    rng.shuffle(context_ids)

    return context_ids


rows = []

for pilot_number, (original_idx, row) in enumerate(pilot.iterrows()):

    question = row["question"]

    # Correct embedding corresponding to this training question
    query_embedding = train_question_embeddings[original_idx]

    # -----------------------------------------
    # Retrieve using all three strategies
    # -----------------------------------------

    retrievals = {
        "bm25": [
            x["chunk_id"]
            for x in retrieve_bm25(question, TOP_K)
        ],

        "dense": [
            x["chunk_id"]
            for x in retrieve_dense(
                query_embedding,
                TOP_K
            )
        ],

        "hybrid": [
            x["chunk_id"]
            for x in retrieve_hybrid(
                question,
                query_embedding,
                TOP_K,
                alpha=0.5
            )
        ]
    }

    # -----------------------------------------
    # Create four noise conditions
    # -----------------------------------------

    for method, retrieved_ids in retrievals.items():

        for noise in NOISE_LEVELS:

            context_ids = build_noisy_context(
                retrieved_ids,
                noise,
                seed=1000 + pilot_number
            )

            rows.append({
                "question_id": row["question_id"],
                "question": question,
                "method": method,
                "noise_level": noise,
                "retrieved_ids": retrieved_ids,
                "context_ids": context_ids,
                "reference_answer": row["answer"]
            })


pilot_contexts = pd.DataFrame(rows)

print("Pilot questions:", PILOT_SIZE)
print("Retrieval methods:", 3)
print("Noise levels:", len(NOISE_LEVELS))
print("Total experimental contexts:", len(pilot_contexts))

print("\nExpected contexts:")
print(
    PILOT_SIZE,
    "×",
    3,
    "×",
    len(NOISE_LEVELS),
    "=",
    PILOT_SIZE * 3 * len(NOISE_LEVELS)
)

print("\nNoise distribution:")
print(
    pilot_contexts["noise_level"]
    .value_counts()
    .sort_index()
)

print("\nMethod distribution:")
print(
    pilot_contexts["method"]
    .value_counts()
)

print("\nSample contexts:")
print(
    pilot_contexts[
        [
            "question_id",
            "method",
            "noise_level",
            "context_ids"
        ]
    ].head(12).to_string(index=False)
)

Pilot questions: 20
Retrieval methods: 3
Noise levels: 4
Total experimental contexts: 240

Expected contexts:
20 × 3 × 4 = 240

Noise distribution:
noise_level
0.00    60
0.25    60
0.50    60
0.75    60
Name: count, dtype: int64

Method distribution:
method
bm25      80
dense     80
hybrid    80
Name: count, dtype: int64

Sample contexts:
question_id method  noise_level                                                                                context_ids
     q_1452   bm25         0.00 [doc_0646_chunk_1, doc_0114_chunk_1, doc_0103_chunk_1, doc_0610_chunk_1, doc_0656_chunk_1]
     q_1452   bm25         0.25 [doc_1064_chunk_1, doc_0114_chunk_1, doc_0103_chunk_1, doc_0610_chunk_1, doc_0656_chunk_1]
     q_1452   bm25         0.50 [doc_0587_chunk_2, doc_0114_chunk_1, doc_0103_chunk_1, doc_0610_chunk_1, doc_1064_chunk_1]
     q_1452   bm25         0.75 [doc_1040_chunk_3, doc_0587_chunk_2, doc_1064_chunk_1, doc_0610_chunk_1, doc_0917_chunk_1]
     q_1452  dense         0.00 [doc_0710_c

In [25]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 4.5 MB/s eta 0:00:00


In [26]:
from groq import Groq
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

GROQ_API_KEY = user_secrets.get_secret("GROQ_API_KEY")

groq_client = Groq(
    api_key=GROQ_API_KEY,
    max_retries=0
)

print("Groq client initialized successfully.")
print("Automatic Groq retries: DISABLED")

Groq client initialized successfully.
Automatic Groq retries: DISABLED


In [27]:
import requests

r = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={
        "Authorization": f"Bearer {GROQ_API_KEY}"
    },
    timeout=20
)

print("Status:", r.status_code)
print("Response:", r.text[:1000])

Status: 200
Response: {"object":"list","data":[{"id":"qwen/qwen3.8-27b","object":"model","created":1786984846,"owned_by":"Alibaba Cloud","active":true,"context_window":131072,"public_apps":null,"max_completion_tokens":16384,"name":"Qwen/Qwen3.8-27B","input_modalities":["text","image"],"output_modalities":["text"],"context_length":131072,"max_output_length":16384,"pricing":{"prompt":"0.0000008","completion":"0.000004","image":"0","request":"0","input_cache_read":"0.0000004"},"supported_sampling_parameters":["temperature","top_p","stop","seed","max_tokens"],"supported_features":["tools","json_mode","reasoning"]},{"id":"openai/gpt-oss-20b","object":"model","created":1754407957,"owned_by":"OpenAI","active":true,"context_window":131072,"public_apps":null,"max_completion_tokens":65536,"hugging_face_id":"openai/gpt-oss-20b","name":"GPT OSS 20B","input_modalities":["text"],"output_modalities":["text"],"context_length":131072,"max_output_length":65536,"pricing":{"prompt":"0.000000075","completi

In [28]:
from groq import Groq
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
GROQ_API_KEY = user_secrets.get_secret("GROQ_API_KEY")

groq_client = Groq(
    api_key=GROQ_API_KEY,
    max_retries=0
)

print("Testing openai/gpt-oss-20b...")

try:
    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "user", "content": "Reply with exactly: OK"}
        ],
        max_tokens=5
    )

    print("SUCCESS")
    print("Response:", response.choices[0].message.content)

except Exception as e:
    print("FAILED")
    print(type(e).__name__)
    print(str(e))

Testing openai/gpt-oss-20b...
SUCCESS
Response: 


In [29]:
# =========================================================
# Helper: retrieve chunk text by chunk_id
# =========================================================

def get_chunk_text(chunk_id):
    match = corpus.loc[
        corpus["chunk_id"] == chunk_id,
        "chunk_text"
    ]

    if match.empty:
        raise KeyError(
            f"Chunk ID not found in corpus: {chunk_id}"
        )

    return match.iloc[0]


print("get_chunk_text() ready.")

get_chunk_text() ready.


In [30]:
# ============================================================
# SINGLE GROQ GENERATION PILOT
# ============================================================

test_row = pilot_contexts.iloc[0]

context_text = "\n\n".join(
    [
        f"[{chunk_id}]\n{get_chunk_text(chunk_id)}"
        for chunk_id in test_row["context_ids"]
    ]
)

prompt = f"""
You are answering a question using ONLY the provided evidence.

Question:
{test_row["question"]}

Evidence:
{context_text}

Instructions:
- Answer the question directly.
- Use only information supported by the evidence.
- Do not invent facts.
- If the evidence is insufficient, explicitly say so.

Answer:
"""

response = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0
)

generated_answer = response.choices[0].message.content.strip()

print("=" * 80)
print("Question:")
print(test_row["question"])

print("\nReference answer:")
print(test_row["reference_answer"])

print("\nGenerated answer:")
print(generated_answer)

print("\nMethod:", test_row["method"])
print("Noise level:", test_row["noise_level"])
print("Context chunks:", test_row["context_ids"])

Question:
Combine evidence from 'EU playback_error incident report (2026-03-26)' and 'Deployment Log: operational notes'. What is the recommended mitigation approach?

Reference answer:
Initial triage reviewed dashboards and logs to reduce uncertainty. Use canary checks after applying changes, and prefer reversible actions under high uncertainty.

Generated answer:
The provided evidence does not include the EU playback_error incident report (2026‑03‑26) or the Deployment Log: operational notes, so I cannot determine the recommended mitigation approach.

Method: bm25
Noise level: 0.0
Context chunks: ['doc_0646_chunk_1', 'doc_0114_chunk_1', 'doc_0103_chunk_1', 'doc_0610_chunk_1', 'doc_0656_chunk_1']


In [31]:
test_question = test_row["question"]

print("=" * 80)
print("QUESTION")
print(test_question)

print("\nREFERENCE ANSWER")
print(test_row["reference_answer"])

print("\nBM25 RETRIEVED EVIDENCE")
print("=" * 80)

for rank, chunk_id in enumerate(test_row["context_ids"], start=1):
    text = get_chunk_text(chunk_id)

    print(f"\n--- Rank {rank}: {chunk_id} ---")
    print(text)

print("\n\nGOLD EVIDENCE")
print("=" * 80)

gold_ids = get_gold_ids(
    train.loc[
        train["question"] == test_question,
        "gold_chunk_ids"
    ].iloc[0]
)

for chunk_id in gold_ids:
    print(f"\n--- {chunk_id} ---")
    print(get_chunk_text(chunk_id))

QUESTION
Combine evidence from 'EU playback_error incident report (2026-03-26)' and 'Deployment Log: operational notes'. What is the recommended mitigation approach?

REFERENCE ANSWER
Initial triage reviewed dashboards and logs to reduce uncertainty. Use canary checks after applying changes, and prefer reversible actions under high uncertainty.

BM25 RETRIEVED EVIDENCE

--- Rank 1: doc_0646_chunk_1 ---
On 2026-10-30, the playback_error impacted the LATAM region with severity 1. Customer symptoms included buffering, retries, and elevated error rates.

--- Rank 2: doc_0114_chunk_1 ---
On 2026-04-12, the playback_error impacted the NA region with severity 2. Customer symptoms included buffering, retries, and elevated error rates.

--- Rank 3: doc_0103_chunk_1 ---
On 2025-05-24, the playback_error impacted the NA region with severity 1. Customer symptoms included buffering, retries, and elevated error rates.

--- Rank 4: doc_0610_chunk_1 ---
On 2026-05-29, the playback_error impacted the L

In [32]:
# ============================================================
# CLEAN CONTROLLED NOISE EXPERIMENT
# Gold evidence is ALWAYS retained.
# Irrelevant chunks are progressively added.
# ============================================================

import random
import pandas as pd

PILOT_SIZE = 20

# Number of irrelevant chunks added to the gold evidence
NOISE_COUNTS = [0, 1, 2, 4]

pilot = train.sample(
    PILOT_SIZE,
    random_state=42
).copy()


def get_random_noise_chunks(exclude_ids, n, seed):
    """Select unrelated chunks from the corpus."""

    exclude_ids = set(exclude_ids)

    available = [
        cid for cid in chunk_ids
        if cid not in exclude_ids
    ]

    rng = random.Random(seed)

    return rng.sample(
        available,
        n
    )


def build_noise_context(gold_ids, n_noise, seed):
    """
    Keep every gold chunk and add exactly n_noise
    unrelated chunks.
    """

    gold_ids = list(dict.fromkeys(gold_ids))

    noise_ids = get_random_noise_chunks(
        exclude_ids=gold_ids,
        n=n_noise,
        seed=seed
    )

    context_ids = gold_ids + noise_ids

    # Randomize ordering so gold evidence is not
    # systematically placed first.
    rng = random.Random(seed)
    rng.shuffle(context_ids)

    total_chunks = len(context_ids)

    noise_ratio = (
        n_noise / total_chunks
        if total_chunks > 0
        else 0
    )

    return context_ids, noise_ratio


rows = []

for pilot_number, (original_idx, row) in enumerate(
    pilot.iterrows()
):

    gold_ids = get_gold_ids(
        row["gold_chunk_ids"]
    )

    for n_noise in NOISE_COUNTS:

        context_ids, noise_ratio = (
            build_noise_context(
                gold_ids=gold_ids,
                n_noise=n_noise,
                seed=3000 + pilot_number
            )
        )

        rows.append({
            "question_id": row["question_id"],
            "question": row["question"],
            "gold_ids": gold_ids,
            "n_gold": len(gold_ids),
            "n_noise": n_noise,
            "total_chunks": len(context_ids),
            "noise_ratio": noise_ratio,
            "context_ids": context_ids,
            "reference_answer": row["answer"]
        })


clean_noise_pilot = pd.DataFrame(rows)


print("Questions:", PILOT_SIZE)
print("Noise conditions:", len(NOISE_COUNTS))
print(
    "Total contexts:",
    len(clean_noise_pilot)
)

print("\nExpected:")
print(
    PILOT_SIZE,
    "×",
    len(NOISE_COUNTS),
    "=",
    PILOT_SIZE * len(NOISE_COUNTS)
)

print("\nNoise conditions:")
print(
    clean_noise_pilot[
        [
            "n_noise",
            "total_chunks",
            "noise_ratio"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["n_noise", "total_chunks"]
    )
    .to_string(index=False)
)

print("\nSample:")
print(
    clean_noise_pilot[
        [
            "question_id",
            "n_gold",
            "n_noise",
            "total_chunks",
            "noise_ratio",
            "context_ids"
        ]
    ]
    .head(12)
    .to_string(index=False)
)

Questions: 20
Noise conditions: 4
Total contexts: 80

Expected:
20 × 4 = 80

Noise conditions:
 n_noise  total_chunks  noise_ratio
       0             1     0.000000
       0             2     0.000000
       1             2     0.500000
       1             3     0.333333
       2             3     0.666667
       2             4     0.500000
       4             5     0.800000
       4             6     0.666667

Sample:
question_id  n_gold  n_noise  total_chunks  noise_ratio                                                                                                  context_ids
     q_1452       2        0             2     0.000000                                                                         [doc_0007_chunk_3, doc_0730_chunk_2]
     q_1452       2        1             3     0.333333                                                       [doc_0007_chunk_3, doc_0730_chunk_2, doc_0873_chunk_3]
     q_1452       2        2             4     0.500000                      

In [33]:
# ============================================================
# GROQ CONTROLLED-NOISE PILOT
# One question, four noise conditions
# ============================================================

test_question_id = "q_1452"

test_cases = clean_noise_pilot[
    clean_noise_pilot["question_id"] == test_question_id
].sort_values("n_noise")


def generate_rag_answer(question, context_ids):

    context_text = "\n\n".join(
        [
            f"[{chunk_id}]\n{get_chunk_text(chunk_id)}"
            for chunk_id in context_ids
        ]
    )

    prompt = f"""
You are answering a question using ONLY the provided evidence.

Question:
{question}

Evidence:
{context_text}

Instructions:
- Answer the question directly.
- Combine evidence when the question requires multiple sources.
- Use only information supported by the evidence.
- Do not invent facts.
- If the evidence is insufficient, say so.

Answer:
"""

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip()


for _, row in test_cases.iterrows():

    answer = generate_rag_answer(
        question=row["question"],
        context_ids=row["context_ids"]
    )

    print("=" * 80)
    print("Noise chunks:", row["n_noise"])
    print("Total chunks:", row["total_chunks"])
    print("Noise ratio:", round(row["noise_ratio"], 3))

    print("\nQuestion:")
    print(row["question"])

    print("\nReference answer:")
    print(row["reference_answer"])

    print("\nGenerated answer:")
    print(answer)

Noise chunks: 0
Total chunks: 2
Noise ratio: 0.0

Question:
Combine evidence from 'EU playback_error incident report (2026-03-26)' and 'Deployment Log: operational notes'. What is the recommended mitigation approach?

Reference answer:
Initial triage reviewed dashboards and logs to reduce uncertainty. Use canary checks after applying changes, and prefer reversible actions under high uncertainty.

Generated answer:
Recommended mitigation approach:

- After applying changes, run **canary checks** to validate the fix in a small, controlled environment.  
- When uncertainty is high, prefer **reversible actions** so that any unintended impact can be quickly undone.  
- **Document the outcomes** of the canary test and the change, and share a concise incident summary with stakeholders for visibility.  
- During initial triage, review dashboards and logs to reduce uncertainty and confirm that the root cause (dependency_timeout) has been addressed.
Noise chunks: 1
Total chunks: 3
Noise ratio: 0

In [34]:
# =========================================================
# RESTORE QUESTION EMBEDDING MAP
# Required by RETRIEVAL STRATEGY × NOISE EXPERIMENT
# No LLM/API calls
# =========================================================

import numpy as np

question_embedding_map = {
    qid: emb
    for qid, emb in zip(
        train["question_id"],
        train_question_embeddings
    )
}

print(
    "Question embeddings:",
    len(question_embedding_map)
)

print(
    "Training questions:",
    len(train)
)

# ---------------------------------------------------------
# Validation
# ---------------------------------------------------------

assert len(question_embedding_map) == len(train), (
    "Embedding map size does not match training data."
)

# Check the first question
first_qid = train.iloc[0]["question_id"]

assert first_qid in question_embedding_map, (
    f"Missing embedding for {first_qid}"
)

print(
    f"Embedding found for {first_qid}:",
    np.asarray(
        question_embedding_map[first_qid]
    ).shape
)

print("\nQuestion embedding map: READY")

Question embeddings: 2200
Training questions: 2200
Embedding found for q_0001: (384,)

Question embedding map: READY


In [35]:
# =========================================================
# RETRIEVAL STRATEGY × NOISE EXPERIMENT
# 20 questions × 3 retrieval methods × 4 noise levels
# = 240 cases
#
# CONSTRUCTION + INSPECTION ONLY.
# No LLM/API calls.
# =========================================================

import random
import pandas as pd
import numpy as np

# =========================================================
# 1. Experiment configuration
# =========================================================

RETRIEVAL_METHODS = [
    "bm25",
    "dense",
    "hybrid"
]

NOISE_COUNTS = [0, 1, 2, 4]

N_QUESTIONS = 20
RETRIEVAL_K = 5

PILOT_PKL = "/kaggle/working/retrieval_noise_pilot.pkl"
PILOT_CSV = "/kaggle/working/retrieval_noise_pilot.csv"


# =========================================================
# 2. Use the same 20 research questions
# =========================================================

research_questions = train.iloc[:N_QUESTIONS].copy()

print("Questions:", len(research_questions))
print("Retrieval methods:", RETRIEVAL_METHODS)
print("Noise levels:", NOISE_COUNTS)

expected_cases = (
    len(research_questions)
    * len(RETRIEVAL_METHODS)
    * len(NOISE_COUNTS)
)

print("Expected cases:", expected_cases)


# =========================================================
# 3. Helper: retrieve using one of the three methods
# =========================================================

def retrieve_for_experiment(
    question,
    question_embedding,
    method,
    k=RETRIEVAL_K
):

    if method == "bm25":

        return retrieve_bm25(
            question,
            k=k
        )

    elif method == "dense":

        return retrieve_dense(
            question_embedding,
            k=k
        )

    elif method == "hybrid":

        return retrieve_hybrid(
            question,
            question_embedding,
            k=k,
            alpha=0.50
        )

    else:

        raise ValueError(
            f"Unknown retrieval method: {method}"
        )


# =========================================================
# 4. Build experiment cases
# =========================================================

experiment_cases = []

# Fixed seed makes distractor selection reproducible.
rng = random.Random(42)


for _, row in research_questions.iterrows():

    question_id = row["question_id"]
    question = row["question"]
    reference_answer = row["answer"]

    # -----------------------------------------------------
    # Get precomputed query embedding
    # -----------------------------------------------------

    query_embedding = question_embedding_map[
        question_id
    ]

    # -----------------------------------------------------
    # Run each retrieval strategy
    # -----------------------------------------------------

    for method in RETRIEVAL_METHODS:

        retrieved = retrieve_for_experiment(
            question=question,
            question_embedding=query_embedding,
            method=method,
            k=RETRIEVAL_K
        )

        retrieved_ids = [
            item["chunk_id"]
            for item in retrieved
        ]

        retrieved_id_set = set(
            retrieved_ids
        )

        # -------------------------------------------------
        # Candidate distractors:
        # chunks NOT already retrieved
        # -------------------------------------------------

        available_noise = [
            cid
            for cid in chunk_ids
            if cid not in retrieved_id_set
        ]

        # Safety check
        if len(available_noise) < max(NOISE_COUNTS):

            raise ValueError(
                f"Not enough available distractors for "
                f"{question_id} / {method}. "
                f"Available={len(available_noise)}, "
                f"Required={max(NOISE_COUNTS)}"
            )

        # -------------------------------------------------
        # Construct each noise condition
        # -------------------------------------------------

        for n_noise in NOISE_COUNTS:

            if n_noise == 0:

                noise_ids = []

            else:

                noise_ids = rng.sample(
                    available_noise,
                    n_noise
                )

            context_ids = (
                retrieved_ids
                + noise_ids
            )

            total_chunks = len(
                context_ids
            )

            noise_ratio = (
                n_noise / total_chunks
            )

            experiment_cases.append({

                "question_id":
                    question_id,

                "question":
                    question,

                "reference_answer":
                    reference_answer,

                "retrieval_method":
                    method,

                "n_noise":
                    n_noise,

                "retrieved_ids":
                    retrieved_ids,

                "noise_ids":
                    noise_ids,

                "context_ids":
                    context_ids,

                "retrieved_count":
                    len(retrieved_ids),

                "total_chunks":
                    total_chunks,

                "noise_ratio":
                    noise_ratio

            })


# =========================================================
# 5. Convert to DataFrame
# =========================================================

retrieval_noise_pilot = pd.DataFrame(
    experiment_cases
)


# =========================================================
# 6. Basic validation
# =========================================================

print("\n" + "=" * 70)
print("EXPERIMENT CONSTRUCTION")
print("=" * 70)

print(
    "Total cases:",
    len(retrieval_noise_pilot)
)

print(
    "Unique questions:",
    retrieval_noise_pilot[
        "question_id"
    ].nunique()
)

print(
    "Methods:",
    retrieval_noise_pilot[
        "retrieval_method"
    ].unique()
)

print(
    "Noise levels:",
    sorted(
        retrieval_noise_pilot[
            "n_noise"
        ].unique()
    )
)


# =========================================================
# 7. Validate expected case count
# =========================================================

assert len(retrieval_noise_pilot) == expected_cases, (
    f"Expected {expected_cases} cases, "
    f"got {len(retrieval_noise_pilot)}"
)

assert (
    retrieval_noise_pilot[
        "question_id"
    ].nunique()
    == N_QUESTIONS
)

assert set(
    retrieval_noise_pilot[
        "retrieval_method"
    ].unique()
) == set(RETRIEVAL_METHODS)

assert set(
    retrieval_noise_pilot[
        "n_noise"
    ].unique()
) == set(NOISE_COUNTS)

print("\nBasic validation: PASSED")


# =========================================================
# 8. Check method × noise distribution
# =========================================================

distribution = (
    retrieval_noise_pilot
    .groupby(
        [
            "retrieval_method",
            "n_noise"
        ]
    )
    .size()
    .reset_index(
        name="cases"
    )
)

print("\nDistribution:")
display(distribution)


# =========================================================
# 9. Verify every combination has 20 questions
# =========================================================

expected_per_condition = N_QUESTIONS

assert (
    distribution["cases"]
    == expected_per_condition
).all(), (
    "Some method × noise condition "
    "does not contain exactly 20 cases."
)

print(
    "\nMethod × noise validation: PASSED"
)


# =========================================================
# 10. Check for duplicate experiment keys
# =========================================================

duplicate_keys = (
    retrieval_noise_pilot
    .groupby(
        [
            "question_id",
            "retrieval_method",
            "n_noise"
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
)

duplicate_keys = duplicate_keys[
    duplicate_keys["count"] > 1
]

print(
    "\nDuplicate experiment keys:"
)

display(
    duplicate_keys
)

assert len(duplicate_keys) == 0, (
    "Duplicate experiment keys detected."
)

print(
    "Duplicate-key validation: PASSED"
)


# =========================================================
# 11. Check noise values
# =========================================================

unexpected_noise = sorted(
    set(
        retrieval_noise_pilot[
            "n_noise"
        ]
    )
    -
    set(NOISE_COUNTS)
)

print(
    "\nUnexpected noise values:",
    unexpected_noise
)

assert len(unexpected_noise) == 0


# =========================================================
# 12. Check context construction
# =========================================================

for _, row in retrieval_noise_pilot.iterrows():

    assert (
        len(row["retrieved_ids"])
        == RETRIEVAL_K
    )

    assert (
        len(row["noise_ids"])
        == row["n_noise"]
    )

    assert (
        len(row["context_ids"])
        == row["total_chunks"]
    )

    # Noise must not overlap retrieved chunks
    assert (
        set(row["retrieved_ids"])
        .isdisjoint(
            set(row["noise_ids"])
        )
    )

print(
    "Context construction validation: PASSED"
)


# =========================================================
# 13. Inspect representative cases
# =========================================================

print("\n" + "=" * 70)
print("REPRESENTATIVE CASES")
print("=" * 70)


for method in RETRIEVAL_METHODS:

    print(
        f"\n--- {method.upper()} ---"
    )

    method_cases = retrieval_noise_pilot[
        (
            retrieval_noise_pilot[
                "retrieval_method"
            ]
            == method
        )
        &
        (
            retrieval_noise_pilot[
                "n_noise"
            ]
            == 4
        )
    ]

    sample = method_cases.iloc[0]

    print(
        "\nQuestion:",
        sample["question"]
    )

    print(
        "\nRetrieved chunks:"
    )

    for cid in sample["retrieved_ids"]:

        print(
            f"\n[{cid}]"
        )

        print(
            get_chunk_text(cid)[:500]
        )

    print(
        "\nRandom distractors:"
    )

    for cid in sample["noise_ids"]:

        print(
            f"\n[{cid}]"
        )

        print(
            get_chunk_text(cid)[:300]
        )

    print(
        "\nTotal chunks:",
        sample["total_chunks"]
    )

    print(
        "Noise ratio:",
        round(
            sample["noise_ratio"],
            3
        )
    )


# =========================================================
# 14. Save experiment definition
# =========================================================

retrieval_noise_pilot.to_pickle(
    PILOT_PKL
)

retrieval_noise_pilot.to_csv(
    PILOT_CSV,
    index=False
)


# =========================================================
# 15. Final confirmation
# =========================================================

print("\n" + "=" * 70)
print("CONSTRUCTION COMPLETE")
print("=" * 70)

print(
    "Total cases:",
    len(retrieval_noise_pilot)
)

print(
    "Expected:",
    expected_cases
)

print(
    "\nSaved construction files:"
)

print(
    f"- {PILOT_PKL}"
)

print(
    f"- {PILOT_CSV}"
)

print(
    "\nPilot file exists:",
    __import__("os").path.exists(PILOT_PKL)
)

print(
    "CSV file exists:",
    __import__("os").path.exists(PILOT_CSV)
)

Questions: 20
Retrieval methods: ['bm25', 'dense', 'hybrid']
Noise levels: [0, 1, 2, 4]
Expected cases: 240

EXPERIMENT CONSTRUCTION
Total cases: 240
Unique questions: 20
Methods: ['bm25' 'dense' 'hybrid']
Noise levels: [np.int64(0), np.int64(1), np.int64(2), np.int64(4)]

Basic validation: PASSED

Distribution:


,retrieval_method,n_noise,cases
0,bm25,0,20
1,bm25,1,20
2,bm25,2,20
3,bm25,4,20
4,dense,0,20
5,dense,1,20
6,dense,2,20
7,dense,4,20
8,hybrid,0,20
9,hybrid,1,20



Method × noise validation: PASSED

Duplicate experiment keys:


,question_id,retrieval_method,n_noise,count


Duplicate-key validation: PASSED

Unexpected noise values: []
Context construction validation: PASSED

REPRESENTATIVE CASES

--- BM25 ---

Question: Based on the knowledge base, summarize the key point from: Architecture note: Playback API / Observability / Search / Billing.

Retrieved chunks:

[doc_0154_chunk_1]
The streaming platform consists of core services including Playback API, Observability, Search, Billing. Requests flow from edge caches to regional clusters with automated health checks.

[doc_0332_chunk_1]
The streaming platform consists of core services including Recommendations, Playback API, CDN, Billing. Requests flow from edge caches to regional clusters with automated health checks.

[doc_0702_chunk_1]
The streaming platform consists of core services including Playback API, Observability, Recommendations, Billing. Requests flow from edge caches to regional clusters with automated health checks.

[doc_0376_chunk_1]
The streaming platform consists of core services includi

In [36]:
print("question_embedding_map" in globals())
print("retrieve_bm25" in globals())
print("retrieve_dense" in globals())
print("retrieve_hybrid" in globals())
print("get_chunk_text" in globals())

True
True
True
True
True


In [37]:
# =========================================================
# RETRIEVAL STRATEGY × NOISE EXPERIMENT
# RESUMABLE GENERATION + EVALUATION
#
# 20 questions × 3 retrieval methods × 4 noise levels
# = 240 total cases
#
# IMPORTANT:
# - Uses retrieval_noise_pilot as the experiment definition
# - Uses retrieval_noise_results.csv as the checkpoint
# - Resumes from existing completed cases
# - Does NOT restart completed cases
# - Saves after every successful case
# - Stops safely on Groq rate limit
# =========================================================

import re
import time
import os
import pandas as pd
import numpy as np
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from groq import RateLimitError


# =========================================================
# 1. Text normalization
# =========================================================

def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9_]+", " ", text)
    return text.split()


# =========================================================
# 2. Token F1
# =========================================================

def token_f1(prediction, reference):

    pred_tokens = normalize_text(prediction)
    ref_tokens = normalize_text(reference)

    if not pred_tokens or not ref_tokens:
        return 0.0

    pred_counts = Counter(pred_tokens)
    ref_counts = Counter(ref_tokens)

    overlap = sum(
        (pred_counts & ref_counts).values()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)

    return (
        2 * precision * recall
        / (precision + recall)
    )


# =========================================================
# 3. Semantic similarity model
# =========================================================

print("Loading evaluation model...")

eval_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Evaluation model ready.")


# =========================================================
# 4. Semantic similarity
# =========================================================

def semantic_similarity(prediction, reference):

    embeddings = eval_model.encode(
        [str(prediction), str(reference)],
        normalize_embeddings=True
    )

    return float(
        cosine_similarity(
            embeddings[0].reshape(1, -1),
            embeddings[1].reshape(1, -1)
        )[0][0]
    )


# =========================================================
# 5. Paths
# =========================================================

PILOT_PATH = (
    "/kaggle/working/retrieval_noise_pilot.pkl"
)

RESULT_FILE = (
    "/kaggle/working/retrieval_noise_results.csv"
)


# =========================================================
# 6. Load experiment definition
# =========================================================

if not os.path.exists(PILOT_PATH):

    raise FileNotFoundError(
        f"""
Experiment definition not found:

{PILOT_PATH}

Run the RETRIEVAL STRATEGY × NOISE
CONSTRUCTION cell first.
"""
    )

retrieval_noise_pilot = pd.read_pickle(
    PILOT_PATH
)

print(
    f"Experiment cases defined: "
    f"{len(retrieval_noise_pilot)}"
)


# =========================================================
# 7. Load existing checkpoint
# =========================================================

if os.path.exists(RESULT_FILE):

    results = pd.read_csv(
        RESULT_FILE
    )

    print(
        f"Loaded existing checkpoint: "
        f"{len(results)} rows"
    )

else:

    results = pd.DataFrame()

    print(
        "No existing checkpoint found."
    )


# =========================================================
# 8. Build completed-case keys
# =========================================================

if len(results) > 0:

    completed_keys = set(
        zip(
            results["question_id"],
            results["retrieval_method"],
            results["n_noise"]
        )
    )

else:

    completed_keys = set()


total_cases = len(
    retrieval_noise_pilot
)

remaining = (
    total_cases
    - len(completed_keys)
)

print("\n" + "=" * 60)
print("EXPERIMENT STATUS")
print("=" * 60)

print(
    f"Total cases:     {total_cases}"
)

print(
    f"Already complete: {len(completed_keys)}"
)

print(
    f"Remaining:       {remaining}"
)


# =========================================================
# 9. Run missing cases
# =========================================================

added_this_run = 0

for _, row in retrieval_noise_pilot.iterrows():

    key = (
        row["question_id"],
        row["retrieval_method"],
        row["n_noise"]
    )

    # -----------------------------------------------------
    # Skip completed cases
    # -----------------------------------------------------

    if key in completed_keys:

        continue


    print("\n" + "-" * 60)

    print(
        f"Case: "
        f"{row['question_id']} | "
        f"{row['retrieval_method']} | "
        f"noise={row['n_noise']}"
    )


    # -----------------------------------------------------
    # Generate answer
    # -----------------------------------------------------

    try:

        start_time = time.time()

        generated_answer = (
            generate_rag_answer(
                question=row["question"],
                context_ids=row["context_ids"]
            )
        )

        latency = (
            time.time() - start_time
        )


    except RateLimitError:

        print(
            "\nRATE LIMIT DETECTED."
        )

        print(
            "Stopping immediately."
        )

        print(
            "Checkpoint is already saved."
        )

        break


    # -----------------------------------------------------
    # Evaluate
    # -----------------------------------------------------

    f1 = token_f1(
        generated_answer,
        row["reference_answer"]
    )

    semantic = semantic_similarity(
        generated_answer,
        row["reference_answer"]
    )


    # -----------------------------------------------------
    # Store result
    # -----------------------------------------------------

    result = {

        "question_id":
            row["question_id"],

        "question":
            row["question"],

        "reference_answer":
            row["reference_answer"],

        "retrieval_method":
            row["retrieval_method"],

        "n_noise":
            row["n_noise"],

        "noise_ratio":
            row["noise_ratio"],

        "retrieved_ids":
            str(row["retrieved_ids"]),

        "noise_ids":
            str(row["noise_ids"]),

        "context_ids":
            str(row["context_ids"]),

        "generated_answer":
            generated_answer,

        "latency_seconds":
            latency,

        "token_f1":
            f1,

        "semantic_similarity":
            semantic
    }


    results = pd.concat(
        [
            results,
            pd.DataFrame([result])
        ],
        ignore_index=True
    )


    completed_keys.add(key)

    added_this_run += 1


    # -----------------------------------------------------
    # SAVE IMMEDIATELY
    # -----------------------------------------------------

    results.to_csv(
        RESULT_FILE,
        index=False
    )


    print(
        f"SUCCESS | "
        f"latency={latency:.2f}s | "
        f"F1={f1:.3f} | "
        f"semantic={semantic:.3f}"
    )

    print(
        f"Saved: {len(results)} / {total_cases}"
    )


    # -----------------------------------------------------
    # Small delay
    # -----------------------------------------------------

    time.sleep(2)


# =========================================================
# 10. Final status
# =========================================================

print("\n" + "=" * 60)
print("UPDATED EXPERIMENT STATUS")
print("=" * 60)

print(
    f"Completed: "
    f"{len(results)} / {total_cases}"
)

print(
    f"Remaining: "
    f"{total_cases - len(results)}"
)

print(
    f"Added this run: "
    f"{added_this_run}"
)

print(
    f"\nCheckpoint:"
)

print(
    RESULT_FILE
)


# =========================================================
# 11. Distribution
# =========================================================

if len(results) > 0:

    print("\nDistribution:")

    distribution = (
        results
        .groupby(
            [
                "retrieval_method",
                "n_noise"
            ]
        )
        .size()
        .reset_index(
            name="count"
        )
    )

    display(
        distribution
    )

Loading evaluation model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluation model ready.
Experiment cases defined: 240
No existing checkpoint found.

EXPERIMENT STATUS
Total cases:     240
Already complete: 0
Remaining:       240

------------------------------------------------------------
Case: q_0001 | bm25 | noise=0
SUCCESS | latency=0.37s | F1=0.033 | semantic=0.334
Saved: 1 / 240

------------------------------------------------------------
Case: q_0001 | bm25 | noise=1
SUCCESS | latency=0.63s | F1=0.087 | semantic=0.297
Saved: 2 / 240

------------------------------------------------------------
Case: q_0001 | bm25 | noise=2
SUCCESS | latency=0.53s | F1=0.136 | semantic=0.280
Saved: 3 / 240

------------------------------------------------------------
Case: q_0001 | bm25 | noise=4
SUCCESS | latency=1.01s | F1=0.093 | semantic=0.282
Saved: 4 / 240

------------------------------------------------------------
Case: q_0001 | dense | noise=0
SUCCESS | latency=0.28s | F1=0.125 | semantic=0.281
Saved: 5 / 240

--------------------------------------

,retrieval_method,n_noise,count
0,bm25,0,2
1,bm25,1,2
2,bm25,2,2
3,bm25,4,1
4,dense,0,1
5,dense,1,1
6,dense,2,1
7,dense,4,1
8,hybrid,0,1
9,hybrid,1,1


In [38]:
# =========================================================
# RETRIEVAL STRATEGY × NOISE EXPERIMENT
# SAFE RESUMABLE GENERATION + EVALUATION
#
# 20 questions × 3 retrieval methods × 4 noise levels
# = 240 total cases
#
# RUN THIS DIRECTLY BELOW THE CONSTRUCTION CELL.
#
# - Uses retrieval_noise_pilot already created above
# - Resumes from retrieval_noise_results.csv if present
# - Saves after EVERY successful case
# - Stops immediately on Groq rate limit
# - Never retries indefinitely
# =========================================================

import os
import time
import re
import pandas as pd
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from groq import RateLimitError


# =========================================================
# 1. Configuration
# =========================================================

RESULT_FILE = "/kaggle/working/retrieval_noise_results.csv"

TOTAL_EXPECTED = 240


# =========================================================
# 2. Make sure construction cell actually ran
# =========================================================

if "retrieval_noise_pilot" not in globals():

    raise RuntimeError(
        """
retrieval_noise_pilot is not loaded.

Run the RETRIEVAL STRATEGY × NOISE
CONSTRUCTION cell first.
"""
    )


if len(retrieval_noise_pilot) != TOTAL_EXPECTED:

    raise ValueError(
        f"Expected {TOTAL_EXPECTED} experiment cases, "
        f"but found {len(retrieval_noise_pilot)}."
    )


print("=" * 70)
print("SAFE RETRIEVAL × NOISE GENERATION")
print("=" * 70)

print(
    f"Experiment definition: "
    f"{len(retrieval_noise_pilot)} cases"
)


# =========================================================
# 3. Load evaluation model
# =========================================================

print("\nLoading evaluation model...")

eval_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Evaluation model ready.")


# =========================================================
# 4. Evaluation helpers
# =========================================================

def normalize_text(text):

    text = str(text).lower()

    text = re.sub(
        r"[^a-z0-9_]+",
        " ",
        text
    )

    return text.split()


def token_f1(prediction, reference):

    pred_tokens = normalize_text(
        prediction
    )

    ref_tokens = normalize_text(
        reference
    )

    if not pred_tokens or not ref_tokens:

        return 0.0

    pred_counts = Counter(
        pred_tokens
    )

    ref_counts = Counter(
        ref_tokens
    )

    overlap = sum(
        (pred_counts & ref_counts).values()
    )

    if overlap == 0:

        return 0.0

    precision = (
        overlap / len(pred_tokens)
    )

    recall = (
        overlap / len(ref_tokens)
    )

    return (
        2 * precision * recall
        / (precision + recall)
    )


def semantic_similarity(
    prediction,
    reference
):

    embeddings = eval_model.encode(
        [
            str(prediction),
            str(reference)
        ],
        normalize_embeddings=True
    )

    return float(
        cosine_similarity(
            embeddings[0].reshape(1, -1),
            embeddings[1].reshape(1, -1)
        )[0][0]
    )


# =========================================================
# 5. Load existing checkpoint
# =========================================================

if os.path.exists(RESULT_FILE):

    results = pd.read_csv(
        RESULT_FILE
    )

    print(
        f"\nExisting checkpoint found: "
        f"{len(results)} rows"
    )

else:

    results = pd.DataFrame()

    print(
        "\nNo existing checkpoint found."
    )


# =========================================================
# 6. Validate checkpoint
# =========================================================

required_columns = [
    "question_id",
    "retrieval_method",
    "n_noise"
]

if len(results) > 0:

    missing_columns = [
        col
        for col in required_columns
        if col not in results.columns
    ]

    if missing_columns:

        raise ValueError(
            "Checkpoint is missing required columns: "
            + str(missing_columns)
        )


# =========================================================
# 7. Build completed experiment keys
# =========================================================

if len(results) > 0:

    completed_keys = set(
        zip(
            results["question_id"],
            results["retrieval_method"],
            results["n_noise"]
        )
    )

else:

    completed_keys = set()


# ---------------------------------------------------------
# Only count keys belonging to the current 240-case pilot
# ---------------------------------------------------------

pilot_keys = set(
    zip(
        retrieval_noise_pilot["question_id"],
        retrieval_noise_pilot["retrieval_method"],
        retrieval_noise_pilot["n_noise"]
    )
)

completed_keys = (
    completed_keys
    & pilot_keys
)


total_cases = len(
    retrieval_noise_pilot
)

completed_count = len(
    completed_keys
)

remaining = (
    total_cases
    - completed_count
)


# =========================================================
# 8. Current status
# =========================================================

print("\n" + "=" * 70)
print("CURRENT STATUS")
print("=" * 70)

print(
    f"Total cases:      {total_cases}"
)

print(
    f"Already complete: {completed_count}"
)

print(
    f"Remaining:        {remaining}"
)


# =========================================================
# 9. Run missing cases
# =========================================================

added_this_run = 0

if remaining == 0:

    print(
        "\nAll 240 cases are already complete."
    )

else:

    for _, row in retrieval_noise_pilot.iterrows():

        key = (
            row["question_id"],
            row["retrieval_method"],
            row["n_noise"]
        )


        # -------------------------------------------------
        # Skip completed cases
        # -------------------------------------------------

        if key in completed_keys:

            continue


        print("\n" + "-" * 70)

        print(
            f"Case "
            f"{completed_count + added_this_run + 1}"
            f"/{total_cases}"
        )

        print(
            f"Question: {row['question_id']}"
        )

        print(
            f"Method: {row['retrieval_method']}"
        )

        print(
            f"Noise: {row['n_noise']}"
        )


        # =================================================
        # Generate answer
        # =================================================

        start_time = time.time()

        try:

            generated_answer = generate_rag_answer(
                question=row["question"],
                context_ids=row["context_ids"]
            )

        except RateLimitError as e:

            print(
                "\n" + "!" * 70
            )

            print(
                "GROQ RATE LIMIT DETECTED"
            )

            print(
                "FULL ERROR DETAILS:"
            )

            print(
                repr(e)
            )

            print(
                "\nERROR MESSAGE:"
            )

            print(
                str(e)
            )

            print(
                "\nStopping immediately."
            )

            print(
                "No retry loop will run."
            )

            print(
                "All previous successful cases "
                "are already saved."
            )

            print(
                "\nCheckpoint:"
            )

            print(
                RESULT_FILE
            )

            print(
                "!" * 70
            )

            break

        except Exception as e:

            print(
                "\nGENERATION ERROR:"
            )

            print(
                repr(e)
            )

            print(
                "\nStopping safely."
            )

            print(
                "All previous successful cases "
                "are already saved."
            )

            break


        latency = (
            time.time()
            - start_time
        )


        # =================================================
        # Evaluate
        # =================================================

        try:

            f1 = token_f1(
                generated_answer,
                row["reference_answer"]
            )

            semantic = semantic_similarity(
                generated_answer,
                row["reference_answer"]
            )

        except Exception as e:

            print(
                "\nEVALUATION ERROR:"
            )

            print(
                repr(e)
            )

            print(
                "Generated answer was NOT "
                "saved as a completed case."
            )

            break


        # =================================================
        # Create result row
        # =================================================

        result = {

            "question_id":
                row["question_id"],

            "question":
                row["question"],

            "reference_answer":
                row["reference_answer"],

            "retrieval_method":
                row["retrieval_method"],

            "n_noise":
                row["n_noise"],

            "noise_ratio":
                row["noise_ratio"],

            "retrieved_ids":
                str(row["retrieved_ids"]),

            "noise_ids":
                str(row["noise_ids"]),

            "context_ids":
                str(row["context_ids"]),

            "generated_answer":
                generated_answer,

            "latency_seconds":
                latency,

            "token_f1":
                f1,

            "semantic_similarity":
                semantic
        }


        # =================================================
        # Append result
        # =================================================

        results = pd.concat(
            [
                results,
                pd.DataFrame([result])
            ],
            ignore_index=True
        )


        completed_keys.add(
            key
        )

        added_this_run += 1


        # =================================================
        # SAVE IMMEDIATELY
        # =================================================

        results.to_csv(
            RESULT_FILE,
            index=False
        )


        print(
            f"SUCCESS | "
            f"latency={latency:.2f}s | "
            f"F1={f1:.3f} | "
            f"semantic={semantic:.3f}"
        )

        print(
            f"Saved: "
            f"{len(results)} / {total_cases}"
        )


        # =================================================
        # Small delay
        # =================================================

        time.sleep(2)


# =========================================================
# 10. Final status
# =========================================================

print("\n" + "=" * 70)
print("EXPERIMENT STATUS")
print("=" * 70)

print(
    f"Completed: "
    f"{len(results)} / {total_cases}"
)

print(
    f"Remaining: "
    f"{total_cases - len(results)}"
)

print(
    f"Added this run: "
    f"{added_this_run}"
)


# =========================================================
# 11. Checkpoint confirmation
# =========================================================

print(
    "\nCheckpoint:"
)

print(
    RESULT_FILE
)

print(
    "Checkpoint exists:",
    os.path.exists(RESULT_FILE)
)


# =========================================================
# 12. Distribution
# =========================================================

if len(results) > 0:

    print("\nDistribution:")

    distribution = (
        results
        .groupby(
            [
                "retrieval_method",
                "n_noise"
            ]
        )
        .size()
        .reset_index(
            name="count"
        )
    )

    display(
        distribution
    )

SAFE RETRIEVAL × NOISE GENERATION
Experiment definition: 240 cases

Loading evaluation model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluation model ready.

Existing checkpoint found: 15 rows

CURRENT STATUS
Total cases:      240
Already complete: 15
Remaining:        225

----------------------------------------------------------------------
Case 16/240
Question: q_0002
Method: bm25
Noise: 4

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
GROQ RATE LIMIT DETECTED
FULL ERROR DETAILS:
RateLimitError("Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01kmav45hhfj88ze7n33xky75w` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7332, Requested 796. Please try again in 960ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}")

ERROR MESSAGE:
Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01kmav45hhfj88ze7n33xky75w` service tier `on_demand` on tokens per minute 

,retrieval_method,n_noise,count
0,bm25,0,2
1,bm25,1,2
2,bm25,2,2
3,bm25,4,1
4,dense,0,1
5,dense,1,1
6,dense,2,1
7,dense,4,1
8,hybrid,0,1
9,hybrid,1,1


In [39]:
# =========================================================
# STATISTICAL TEST OF NOISE EFFECT
# Retrieval strategy × controlled noise
#
# 240 completed cases:
# 20 questions × 3 methods × 4 noise levels
# =========================================================

import pandas as pd
import numpy as np
from scipy.stats import wilcoxon

# ---------------------------------------------------------
# Load the ACTUAL completed 240-case experiment
# ---------------------------------------------------------

RESULT_FILE = "/kaggle/working/retrieval_noise_results.csv"

noise_results = pd.read_csv(RESULT_FILE)

print("=" * 90)
print("LOADED RETRIEVAL × NOISE RESULTS")
print("=" * 90)

print(f"Rows: {len(noise_results)}")
print(
    f"Methods: {sorted(noise_results['retrieval_method'].unique())}"
)
print(
    f"Noise levels: {sorted(noise_results['n_noise'].unique())}"
)

assert len(noise_results) == 240, (
    f"Expected 240 rows, found {len(noise_results)}"
)


# =========================================================
# 1. Paired baseline comparisons
#
# Compare each noisy condition against n_noise = 0
# separately for each retrieval method.
# =========================================================

stats = []

for method in sorted(
    noise_results["retrieval_method"].unique()
):

    method_df = noise_results[
        noise_results["retrieval_method"] == method
    ]

    baseline = (
        method_df[
            method_df["n_noise"] == 0
        ]
        .set_index("question_id")
    )

    for noise_level in [1, 2, 4]:

        noisy = (
            method_df[
                method_df["n_noise"] == noise_level
            ]
            .set_index("question_id")
        )

        common_ids = baseline.index.intersection(
            noisy.index
        )

        baseline_f1 = baseline.loc[
            common_ids,
            "token_f1"
        ].astype(float)

        noisy_f1 = noisy.loc[
            common_ids,
            "token_f1"
        ].astype(float)

        baseline_sem = baseline.loc[
            common_ids,
            "semantic_similarity"
        ].astype(float)

        noisy_sem = noisy.loc[
            common_ids,
            "semantic_similarity"
        ].astype(float)

        # -------------------------------------------------
        # Wilcoxon signed-rank test
        # -------------------------------------------------

        f1_stat, f1_p = wilcoxon(
            baseline_f1,
            noisy_f1,
            alternative="two-sided"
        )

        sem_stat, sem_p = wilcoxon(
            baseline_sem,
            noisy_sem,
            alternative="two-sided"
        )

        baseline_f1_mean = baseline_f1.mean()
        noisy_f1_mean = noisy_f1.mean()

        baseline_sem_mean = baseline_sem.mean()
        noisy_sem_mean = noisy_sem.mean()

        stats.append({

            "retrieval_method":
                method,

            "n_noise":
                noise_level,

            "n":
                len(common_ids),

            # -------------------------
            # Token F1
            # -------------------------

            "baseline_f1":
                baseline_f1_mean,

            "noisy_f1":
                noisy_f1_mean,

            "f1_change":
                noisy_f1_mean
                - baseline_f1_mean,

            "f1_relative_change_pct":
                (
                    (noisy_f1_mean - baseline_f1_mean)
                    / baseline_f1_mean
                    * 100
                ),

            "f1_wilcoxon_stat":
                f1_stat,

            "f1_p_value":
                f1_p,

            # -------------------------
            # Semantic similarity
            # -------------------------

            "baseline_semantic":
                baseline_sem_mean,

            "noisy_semantic":
                noisy_sem_mean,

            "semantic_change":
                noisy_sem_mean
                - baseline_sem_mean,

            "semantic_relative_change_pct":
                (
                    (noisy_sem_mean - baseline_sem_mean)
                    / baseline_sem_mean
                    * 100
                ),

            "semantic_wilcoxon_stat":
                sem_stat,

            "semantic_p_value":
                sem_p
        })


# =========================================================
# 2. Convert to DataFrame
# =========================================================

noise_stats = pd.DataFrame(stats)


# =========================================================
# 3. Multiple-comparison correction
#
# 9 F1 comparisons:
# 3 retrieval methods × 3 noise levels
#
# 9 semantic comparisons:
# 3 retrieval methods × 3 noise levels
# =========================================================

N_TESTS = 9

noise_stats["f1_p_bonferroni"] = np.minimum(
    noise_stats["f1_p_value"] * N_TESTS,
    1.0
)

noise_stats["semantic_p_bonferroni"] = np.minimum(
    noise_stats["semantic_p_value"] * N_TESTS,
    1.0
)

noise_stats["f1_significant"] = (
    noise_stats["f1_p_bonferroni"] < 0.05
)

noise_stats["semantic_significant"] = (
    noise_stats["semantic_p_bonferroni"] < 0.05
)


# =========================================================
# 4. Display results
# =========================================================

print("\n" + "=" * 100)
print("BASELINE VS CONTROLLED NOISE — WILCOXON ANALYSIS")
print("=" * 100)

display(
    noise_stats.round(4)
)


# =========================================================
# 5. Save statistical results
# =========================================================

STATS_FILE = (
    "/kaggle/working/"
    "retrieval_noise_statistics.csv"
)

noise_stats.to_csv(
    STATS_FILE,
    index=False
)

print("\n" + "=" * 90)
print("STATISTICAL ANALYSIS COMPLETE")
print("=" * 90)

print(
    f"Comparisons: {len(noise_stats)}"
)

print(
    f"Saved: {STATS_FILE}"
)

LOADED RETRIEVAL × NOISE RESULTS
Rows: 15
Methods: ['bm25', 'dense', 'hybrid']
Noise levels: [np.int64(0), np.int64(1), np.int64(2), np.int64(4)]


AssertionError: Expected 240 rows, found 15

In [ ]:
# =========================================================
# AGGREGATE RETRIEVAL × NOISE RESULTS
# =========================================================

import pandas as pd
import numpy as np

RESULT_FILE = "/kaggle/working/retrieval_noise_results.csv"

# ---------------------------------------------------------
# Load the actual completed 240-case experiment
# ---------------------------------------------------------

results = pd.read_csv(RESULT_FILE)

print("=" * 70)
print("RETRIEVAL × NOISE RESULTS")
print("=" * 70)

print(f"Loaded rows: {len(results)}")

# Safety check
assert len(results) == 240, (
    f"Expected 240 rows, found {len(results)}"
)

# ---------------------------------------------------------
# Aggregate by retrieval method × noise
# ---------------------------------------------------------

summary = (
    results
    .groupby(
        ["retrieval_method", "n_noise"]
    )
    .agg(
        mean_token_f1=("token_f1", "mean"),
        std_token_f1=("token_f1", "std"),

        mean_semantic_similarity=(
            "semantic_similarity",
            "mean"
        ),

        std_semantic_similarity=(
            "semantic_similarity",
            "std"
        ),

        mean_latency=(
            "latency_seconds",
            "mean"
        ),

        n=("question_id", "count")
    )
    .reset_index()
)

# ---------------------------------------------------------
# Mean noise ratio
# ---------------------------------------------------------

noise_ratio = (
    results
    .groupby(
        ["retrieval_method", "n_noise"]
    )["noise_ratio"]
    .mean()
    .reset_index(
        name="noise_ratio_mean"
    )
)

summary = summary.merge(
    noise_ratio,
    on=[
        "retrieval_method",
        "n_noise"
    ],
    how="left"
)

# ---------------------------------------------------------
# Baseline-relative degradation
# Baseline = same retrieval method with n_noise = 0
# ---------------------------------------------------------

baseline = (
    summary[
        summary["n_noise"] == 0
    ][
        [
            "retrieval_method",
            "mean_token_f1",
            "mean_semantic_similarity"
        ]
    ]
    .rename(
        columns={
            "mean_token_f1":
                "baseline_token_f1",

            "mean_semantic_similarity":
                "baseline_semantic_similarity"
        }
    )
)

summary = summary.merge(
    baseline,
    on="retrieval_method",
    how="left"
)

# ---------------------------------------------------------
# Absolute change from baseline
# ---------------------------------------------------------

summary["f1_change_vs_baseline"] = (
    summary["mean_token_f1"]
    - summary["baseline_token_f1"]
)

summary["semantic_change_vs_baseline"] = (
    summary["mean_semantic_similarity"]
    - summary["baseline_semantic_similarity"]
)

# ---------------------------------------------------------
# Relative percentage change
# ---------------------------------------------------------

summary["f1_relative_change_pct"] = (
    summary["f1_change_vs_baseline"]
    / summary["baseline_token_f1"]
    * 100
)

summary["semantic_relative_change_pct"] = (
    summary["semantic_change_vs_baseline"]
    / summary["baseline_semantic_similarity"]
    * 100
)

# ---------------------------------------------------------
# Display
# ---------------------------------------------------------

display(
    summary.round(4)
)

print("\n" + "=" * 70)
print("AGGREGATION COMPLETE")
print("=" * 70)

In [ ]:
# =========================================================
# AGGREGATE RETRIEVAL × CONTROLLED-NOISE RESULTS
# =========================================================

import pandas as pd
import numpy as np

RESULT_FILE = "/kaggle/working/retrieval_noise_results.csv"

# ---------------------------------------------------------
# 1. Load the completed 240-case experiment
# ---------------------------------------------------------

results = pd.read_csv(RESULT_FILE)

print("=" * 70)
print("LOADED EXPERIMENT RESULTS")
print("=" * 70)

print(f"Rows: {len(results)}")
print(f"Expected: 240")

assert len(results) == 240, (
    f"Expected 240 rows, found {len(results)}"
)

# ---------------------------------------------------------
# 2. Aggregate by noise level
# ---------------------------------------------------------

summary = (
    results
    .groupby("n_noise")
    .agg(
        mean_token_f1=("token_f1", "mean"),
        std_token_f1=("token_f1", "std"),

        mean_semantic_similarity=(
            "semantic_similarity",
            "mean"
        ),

        std_semantic_similarity=(
            "semantic_similarity",
            "std"
        ),

        mean_latency_seconds=(
            "latency_seconds",
            "mean"
        ),

        std_latency_seconds=(
            "latency_seconds",
            "std"
        ),

        n=("question_id", "count")
    )
    .reset_index()
)

# ---------------------------------------------------------
# 3. Mean actual noise ratio
# ---------------------------------------------------------

noise_ratio_summary = (
    results
    .groupby("n_noise")["noise_ratio"]
    .mean()
    .reset_index(
        name="noise_ratio_mean"
    )
)

summary = summary.merge(
    noise_ratio_summary,
    on="n_noise",
    how="left"
)

# ---------------------------------------------------------
# 4. Baseline = zero-noise condition
# ---------------------------------------------------------

baseline_f1 = summary.loc[
    summary["n_noise"] == 0,
    "mean_token_f1"
].iloc[0]

baseline_semantic = summary.loc[
    summary["n_noise"] == 0,
    "mean_semantic_similarity"
].iloc[0]

# ---------------------------------------------------------
# 5. Absolute change vs baseline
# ---------------------------------------------------------

summary["f1_change_vs_baseline"] = (
    summary["mean_token_f1"]
    - baseline_f1
)

summary["semantic_change_vs_baseline"] = (
    summary["mean_semantic_similarity"]
    - baseline_semantic
)

# ---------------------------------------------------------
# 6. Relative change vs baseline (%)
# ---------------------------------------------------------

summary["f1_relative_change_pct"] = (
    (
        summary["mean_token_f1"]
        - baseline_f1
    )
    / baseline_f1
    * 100
)

summary["semantic_relative_change_pct"] = (
    (
        summary["mean_semantic_similarity"]
        - baseline_semantic
    )
    / baseline_semantic
    * 100
)

# ---------------------------------------------------------
# 7. Display
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("CONTROLLED-NOISE SUMMARY")
print("=" * 70)

display(
    summary.round(4)
)

# ---------------------------------------------------------
# 8. Verify every noise condition has 60 cases
# ---------------------------------------------------------

print("\nCases per noise condition:")

noise_counts = (
    results
    .groupby("n_noise")
    .size()
    .reset_index(name="count")
)

display(noise_counts)

assert (
    noise_counts["count"] == 60
).all(), (
    "Each noise level should contain "
    "20 questions × 3 retrieval methods = 60 cases."
)

print("Noise-condition validation: PASSED")

# ---------------------------------------------------------
# 9. Save summary
# ---------------------------------------------------------

SUMMARY_FILE = (
    "/kaggle/working/retrieval_noise_summary.csv"
)

summary.to_csv(
    SUMMARY_FILE,
    index=False
)

print(
    f"\nSummary saved to:\n{SUMMARY_FILE}"
)

In [ ]:
import pandas as pd
import os

RESULTS_PATH = "/kaggle/working/retrieval_noise_results.csv"

if not os.path.exists(RESULTS_PATH):
    raise FileNotFoundError(
        f"Checkpoint not found: {RESULTS_PATH}"
    )

retrieval_results = pd.read_csv(RESULTS_PATH)

print("=" * 60)
print("CURRENT RETRIEVAL × NOISE EXPERIMENT")
print("=" * 60)

print("Completed:", len(retrieval_results), "/ 240")
print("Remaining:", 240 - len(retrieval_results))

print("\nDistribution:")
display(
    retrieval_results
    .groupby(["retrieval_method", "n_noise"])
    .size()
    .unstack(fill_value=0)
)

In [ ]:
import os
import glob

print("=" * 60)
print("KAGGLE WORKING FILES")
print("=" * 60)

for path in sorted(glob.glob("/kaggle/working/*")):
    print(path)

print("\nTarget checkpoint:")
print(
    os.path.exists(
        "/kaggle/working/retrieval_noise_results.csv"
    )
)

In [ ]:
import os
import glob

print("Kaggle working files:")
for f in sorted(glob.glob("/kaggle/working/*")):
    print(f)

print("\nretrieval_noise_results.csv exists:",
      os.path.exists("/kaggle/working/retrieval_noise_results.csv"))

In [ ]:
# =========================================================
# HARD-NOISE EXPERIMENT
# Select semantically similar non-gold chunks as distractors
# =========================================================

import numpy as np
import pandas as pd
import random

HARD_NOISE_COUNTS = [0, 1, 2, 4]

# ---------------------------------------------------------
# Map question_id -> embedding
# ---------------------------------------------------------

question_embedding_map = {
    qid: emb
    for qid, emb in zip(
        train["question_id"],
        train_question_embeddings
    )
}


# ---------------------------------------------------------
# Select hard distractors
# ---------------------------------------------------------

def get_hard_noise_chunks(
    question_id,
    gold_ids,
    n,
    candidate_start=5,
    candidate_end=50,
    seed=42
):
    """
    Select semantically similar non-gold chunks.

    candidate_start=5 avoids taking the very top dense
    candidates, which may contain genuinely relevant evidence.

    candidate_end=50 keeps the distractors relatively
    semantically close to the question.
    """

    if n == 0:
        return []

    query_embedding = np.asarray(
        question_embedding_map[question_id],
        dtype="float32"
    )

    scores = np.dot(
        corpus_embeddings,
        query_embedding
    )

    ranked_indices = np.argsort(scores)[::-1]

    gold_ids = set(gold_ids)

    candidates = []

    for idx in ranked_indices:

        cid = chunk_ids[idx]

        if cid in gold_ids:
            continue

        candidates.append(
            {
                "chunk_id": cid,
                "score": float(scores[idx])
            }
        )

        if len(candidates) >= candidate_end:
            break

    # Use a middle/high similarity region rather than
    # always selecting the absolute closest chunk.
    candidates = candidates[
        candidate_start:candidate_end
    ]

    rng = random.Random(seed)

    if len(candidates) < n:
        raise ValueError(
            f"Only {len(candidates)} hard-noise candidates "
            f"available for {question_id}"
        )

    selected = rng.sample(candidates, n)

    return [
        item["chunk_id"]
        for item in selected
    ]


# ---------------------------------------------------------
# Build contexts
# ---------------------------------------------------------

hard_rows = []

for pilot_number, (original_idx, row) in enumerate(
    pilot.iterrows()
):

    gold_ids = get_gold_ids(
        row["gold_chunk_ids"]
    )

    for n_noise in HARD_NOISE_COUNTS:

        noise_ids = get_hard_noise_chunks(
            question_id=row["question_id"],
            gold_ids=gold_ids,
            n=n_noise,
            seed=5000 + pilot_number
        )

        context_ids = list(gold_ids) + noise_ids

        rng = random.Random(
            6000 + pilot_number
        )

        rng.shuffle(context_ids)

        total_chunks = len(context_ids)

        noise_ratio = (
            n_noise / total_chunks
            if total_chunks > 0
            else 0
        )

        hard_rows.append({
            "question_id": row["question_id"],
            "question": row["question"],
            "gold_ids": gold_ids,
            "n_gold": len(gold_ids),
            "n_noise": n_noise,
            "total_chunks": total_chunks,
            "noise_ratio": noise_ratio,
            "context_ids": context_ids,
            "reference_answer": row["answer"]
        })


hard_noise_pilot = pd.DataFrame(hard_rows)

print(
    "Hard-noise contexts:",
    len(hard_noise_pilot)
)

display(
    hard_noise_pilot[
        [
            "question_id",
            "n_noise",
            "total_chunks",
            "noise_ratio"
        ]
    ].head(20)
)

In [ ]:
# =========================================================
# Inspect hard distractors for several questions
# =========================================================

for question_id in [
    "q_1452",
    "q_2016",
    "q_0180"
]:

    print("\n" + "=" * 90)
    print("QUESTION:", question_id)

    row = hard_noise_pilot[
        (hard_noise_pilot["question_id"] == question_id) &
        (hard_noise_pilot["n_noise"] == 4)
    ].iloc[0]

    print("\nQuestion:")
    print(row["question"])

    print("\nGold chunks:")
    for cid in row["gold_ids"]:
        print(f"\n[{cid}]")
        print(get_chunk_text(cid))

    print("\nHard-noise chunks:")
    for cid in row["context_ids"]:
        if cid not in row["gold_ids"]:
            print(f"\n[{cid}]")
            print(get_chunk_text(cid))

In [ ]:
# =========================================================
# HARD-NOISE GENERATION + EVALUATION
# 20 questions × 4 noise levels = 80 cases
# =========================================================

import time
import pandas as pd
from groq import RateLimitError

HARD_RESULT_FILE = "/kaggle/working/hard_noise_results.csv"

# ---------------------------------------------------------
# Load existing results if the cell was previously run
# ---------------------------------------------------------

try:
    hard_results = pd.read_csv(HARD_RESULT_FILE)

    completed_keys = set(
        zip(
            hard_results["question_id"],
            hard_results["n_noise"]
        )
    )

    print(
        f"Loaded {len(hard_results)} existing hard-noise results."
    )

except FileNotFoundError:

    hard_results = pd.DataFrame()

    completed_keys = set()

    print("Starting hard-noise experiment from scratch.")


# ---------------------------------------------------------
# Run missing cases
# ---------------------------------------------------------

for _, row in hard_noise_pilot.iterrows():

    key = (
        row["question_id"],
        row["n_noise"]
    )

    # Skip already completed cases
    if key in completed_keys:
        continue

    print(
        f"\nRunning: {row['question_id']} | "
        f"Hard noise: {row['n_noise']}"
    )

    # -----------------------------------------------------
    # Generate answer
    # -----------------------------------------------------

    while True:

        try:

            generated_answer = generate_rag_answer(
                question=row["question"],
                context_ids=row["context_ids"]
            )

            break

        except RateLimitError:

            print(
                "Groq rate limit reached. "
                "Waiting 5 seconds..."
            )

            time.sleep(5)

    # -----------------------------------------------------
    # Evaluate
    # -----------------------------------------------------

    f1 = token_f1(
        generated_answer,
        row["reference_answer"]
    )

    semantic = semantic_similarity(
        generated_answer,
        row["reference_answer"]
    )

    result = {
        "question_id": row["question_id"],
        "n_noise": row["n_noise"],
        "total_chunks": row["total_chunks"],
        "noise_ratio": row["noise_ratio"],
        "reference_answer": row["reference_answer"],
        "generated_answer": generated_answer,
        "token_f1": f1,
        "semantic_similarity": semantic
    }

    hard_results = pd.concat(
        [
            hard_results,
            pd.DataFrame([result])
        ],
        ignore_index=True
    )

    completed_keys.add(key)

    # -----------------------------------------------------
    # Save immediately
    # -----------------------------------------------------

    hard_results.to_csv(
        HARD_RESULT_FILE,
        index=False
    )

    print(
        f"F1: {f1:.3f} | "
        f"Semantic similarity: {semantic:.3f}"
    )

    # Avoid hammering the TPM limit
    time.sleep(2)


print("\n" + "=" * 60)
print(
    f"Completed: {len(hard_results)} / 80 hard-noise cases"
)
print("=" * 60)

In [ ]:
# =========================================================
# HARD-NOISE AGGREGATION + RANDOM VS HARD COMPARISON
# =========================================================

import pandas as pd
import numpy as np

# ---------------------------------------------------------
# Load existing experiment results
# ---------------------------------------------------------

random_results = pd.read_csv(
    "/kaggle/working/retrieval_noise_results.csv"
)

hard_results = pd.read_csv(
    "/kaggle/working/hard_noise_results.csv"
)

print("Random-noise cases:", len(random_results))
print("Hard-noise cases:", len(hard_results))

print("\nRandom-noise columns:")
print(random_results.columns.tolist())

print("\nHard-noise columns:")
print(hard_results.columns.tolist())


# ---------------------------------------------------------
# Aggregate results
# ---------------------------------------------------------

def aggregate_results(df, experiment_name):

    summary = (
        df.groupby("n_noise")
        .agg(
            mean_token_f1=("token_f1", "mean"),
            std_token_f1=("token_f1", "std"),
            mean_semantic_similarity=(
                "semantic_similarity",
                "mean"
            ),
            std_semantic_similarity=(
                "semantic_similarity",
                "std"
            ),
            n=("question_id", "count"),
            noise_ratio_mean=("noise_ratio", "mean")
        )
        .reset_index()
    )

    baseline_f1 = summary.loc[
        summary["n_noise"] == 0,
        "mean_token_f1"
    ].iloc[0]

    baseline_semantic = summary.loc[
        summary["n_noise"] == 0,
        "mean_semantic_similarity"
    ].iloc[0]

    summary["f1_change_vs_baseline"] = (
        summary["mean_token_f1"] - baseline_f1
    )

    summary["semantic_change_vs_baseline"] = (
        summary["mean_semantic_similarity"]
        - baseline_semantic
    )

    summary["f1_relative_change_pct"] = (
        summary["f1_change_vs_baseline"]
        / baseline_f1
        * 100
    )

    summary["semantic_relative_change_pct"] = (
        summary["semantic_change_vs_baseline"]
        / baseline_semantic
        * 100
    )

    summary["experiment"] = experiment_name

    return summary


random_summary = aggregate_results(
    random_results,
    "Random distractors"
)

hard_summary = aggregate_results(
    hard_results,
    "Hard semantic distractors"
)


# ---------------------------------------------------------
# Random-noise summary
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("RANDOM DISTRACTOR RESULTS")
print("=" * 70)

display(
    random_summary[
        [
            "n_noise",
            "mean_token_f1",
            "std_token_f1",
            "mean_semantic_similarity",
            "std_semantic_similarity",
            "noise_ratio_mean",
            "f1_relative_change_pct",
            "semantic_relative_change_pct"
        ]
    ].round(4)
)


# ---------------------------------------------------------
# Hard-noise summary
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("HARD SEMANTIC DISTRACTOR RESULTS")
print("=" * 70)

display(
    hard_summary[
        [
            "n_noise",
            "mean_token_f1",
            "std_token_f1",
            "mean_semantic_similarity",
            "std_semantic_similarity",
            "noise_ratio_mean",
            "f1_relative_change_pct",
            "semantic_relative_change_pct"
        ]
    ].round(4)
)


# ---------------------------------------------------------
# Direct comparison
# ---------------------------------------------------------

comparison = pd.merge(
    random_summary[
        [
            "n_noise",
            "mean_token_f1",
            "mean_semantic_similarity"
        ]
    ],
    hard_summary[
        [
            "n_noise",
            "mean_token_f1",
            "mean_semantic_similarity"
        ]
    ],
    on="n_noise",
    suffixes=("_random", "_hard")
)

comparison["f1_hard_minus_random"] = (
    comparison["mean_token_f1_hard"]
    - comparison["mean_token_f1_random"]
)

comparison["semantic_hard_minus_random"] = (
    comparison["mean_semantic_similarity_hard"]
    - comparison["mean_semantic_similarity_random"]
)

comparison["f1_hard_vs_random_pct"] = (
    comparison["f1_hard_minus_random"]
    / comparison["mean_token_f1_random"]
    * 100
)

comparison["semantic_hard_vs_random_pct"] = (
    comparison["semantic_hard_minus_random"]
    / comparison["mean_semantic_similarity_random"]
    * 100
)


print("\n" + "=" * 70)
print("RANDOM VS HARD SEMANTIC DISTRACTORS")
print("=" * 70)

display(comparison.round(4))


# ---------------------------------------------------------
# Save summaries
# ---------------------------------------------------------

random_summary.to_csv(
    "/kaggle/working/random_noise_summary.csv",
    index=False
)

hard_summary.to_csv(
    "/kaggle/working/hard_noise_summary.csv",
    index=False
)

comparison.to_csv(
    "/kaggle/working/random_vs_hard_comparison.csv",
    index=False
)

print("\nSaved:")
print("- random_noise_summary.csv")
print("- hard_noise_summary.csv")
print("- random_vs_hard_comparison.csv")

In [ ]:
import os
import glob

print("CSV files currently in /kaggle/working:\n")

for path in sorted(glob.glob("/kaggle/working/*.csv")):
    print(path)

print("\nHard-noise file exists:",
      os.path.exists("/kaggle/working/hard_noise_results.csv"))

print("Random-noise expected file exists:",
      os.path.exists("/kaggle/working/random_noise_results.csv"))

In [ ]:
# =========================================================
# LOAD COMPLETED RETRIEVAL × NOISE RESULTS
# =========================================================

import os
import pandas as pd

RESULTS_PATH = "/kaggle/working/retrieval_noise_results.csv"

if not os.path.exists(RESULTS_PATH):
    raise FileNotFoundError(
        f"Completed experiment results not found:\n{RESULTS_PATH}"
    )

# Load the actual 240-case experiment
results = pd.read_csv(RESULTS_PATH)

# Use the expected variable name for downstream analysis
noise_results = results.copy()

print("=" * 70)
print("RETRIEVAL × NOISE RESULTS LOADED")
print("=" * 70)

print(f"Rows: {len(noise_results)}")
print(f"Columns: {len(noise_results.columns)}")

print("\nDistribution:")

display(
    noise_results
    .groupby(["retrieval_method", "n_noise"])
    .size()
    .reset_index(name="count")
)

print("\nRequired evaluation columns:")

required = [
    "question_id",
    "retrieval_method",
    "n_noise",
    "noise_ratio",
    "token_f1",
    "semantic_similarity"
]

missing = [
    col for col in required
    if col not in noise_results.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

print("All required columns present.")

print("\nStatus:")
print("240 / 240 cases available.")

In [ ]:
import os

for path in [
    "/kaggle/working/retrieval_noise_results.csv",
    "/kaggle/working/retrieval_noise_pilot.pkl",
    "/kaggle/working/retrieval_noise_pilot.csv",
]:
    print(os.path.basename(path), "=>", os.path.exists(path))

In [ ]:
# =========================================================
# RETRIEVAL STRATEGY × NOISE EXPERIMENT
# 20 questions × 3 retrieval methods × 4 noise levels
# = 240 cases
#
# CONSTRUCTION + INSPECTION ONLY
# No LLM/API calls yet.
# =========================================================

import random
import pandas as pd
import numpy as np

RETRIEVAL_METHODS = [
    "bm25",
    "dense",
    "hybrid"
]

NOISE_COUNTS = [0, 1, 2, 4]

# ---------------------------------------------------------
# Use the same 20 research questions
# ---------------------------------------------------------

research_questions = train.iloc[:20].copy()

print("Questions:", len(research_questions))
print("Retrieval methods:", RETRIEVAL_METHODS)
print("Noise levels:", NOISE_COUNTS)
print(
    "Expected cases:",
    len(research_questions)
    * len(RETRIEVAL_METHODS)
    * len(NOISE_COUNTS)
)


# ---------------------------------------------------------
# Helper: retrieve using one of the three methods
# ---------------------------------------------------------

def retrieve_for_experiment(
    question,
    question_embedding,
    method,
    k=5
):

    if method == "bm25":

        return retrieve_bm25(
            question,
            k=k
        )

    elif method == "dense":

        return retrieve_dense(
            question_embedding,
            k=k
        )

    elif method == "hybrid":

        return retrieve_hybrid(
            question,
            question_embedding,
            k=k,
            alpha=0.50
        )

    else:
        raise ValueError(
            f"Unknown retrieval method: {method}"
        )


# ---------------------------------------------------------
# Build experiment cases
# ---------------------------------------------------------

experiment_cases = []

rng = random.Random(42)

for _, row in research_questions.iterrows():

    question_id = row["question_id"]
    question = row["question"]
    reference_answer = row["answer"]

    query_embedding = question_embedding_map[
        question_id
    ]

    for method in RETRIEVAL_METHODS:

        retrieved = retrieve_for_experiment(
            question=question,
            question_embedding=query_embedding,
            method=method,
            k=5
        )

        retrieved_ids = [
            item["chunk_id"]
            for item in retrieved
        ]

        # -------------------------------------------------
        # Add controlled random distractors
        #
        # Distractors are selected from chunks that are
        # NOT already retrieved by this method.
        # -------------------------------------------------

        available_noise = [
            cid
            for cid in chunk_ids
            if cid not in set(retrieved_ids)
        ]

        for n_noise in NOISE_COUNTS:

            if n_noise == 0:

                noise_ids = []

            else:

                noise_ids = rng.sample(
                    available_noise,
                    n_noise
                )

            context_ids = (
                retrieved_ids
                + noise_ids
            )

            experiment_cases.append({

                "question_id":
                    question_id,

                "question":
                    question,

                "reference_answer":
                    reference_answer,

                "retrieval_method":
                    method,

                "n_noise":
                    n_noise,

                "retrieved_ids":
                    retrieved_ids,

                "noise_ids":
                    noise_ids,

                "context_ids":
                    context_ids,

                "retrieved_count":
                    len(retrieved_ids),

                "total_chunks":
                    len(context_ids),

                "noise_ratio":
                    n_noise / len(context_ids)

            })


retrieval_noise_pilot = pd.DataFrame(
    experiment_cases
)


# ---------------------------------------------------------
# Basic validation
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("EXPERIMENT CONSTRUCTION")
print("=" * 70)

print(
    "Total cases:",
    len(retrieval_noise_pilot)
)

print(
    "Unique questions:",
    retrieval_noise_pilot["question_id"].nunique()
)

print(
    "Methods:",
    retrieval_noise_pilot["retrieval_method"].unique()
)

print(
    "Noise levels:",
    sorted(
        retrieval_noise_pilot["n_noise"].unique()
    )
)


# ---------------------------------------------------------
# Check expected distribution
# ---------------------------------------------------------

distribution = (
    retrieval_noise_pilot
    .groupby(
        [
            "retrieval_method",
            "n_noise"
        ]
    )
    .size()
    .reset_index(
        name="cases"
    )
)

print("\nDistribution:")
display(distribution)


# ---------------------------------------------------------
# Inspect representative cases
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("REPRESENTATIVE CASES")
print("=" * 70)

for method in RETRIEVAL_METHODS:

    print(
        f"\n--- {method.upper()} ---"
    )

    sample = retrieval_noise_pilot[
        (
            retrieval_noise_pilot[
                "retrieval_method"
            ] == method
        )
        &
        (
            retrieval_noise_pilot[
                "n_noise"
            ] == 4
        )
    ].iloc[0]

    print(
        "Question:",
        sample["question"]
    )

    print(
        "\nRetrieved chunks:"
    )

    for cid in sample["retrieved_ids"]:
        print(
            f"\n[{cid}]"
        )
        print(
            get_chunk_text(cid)[:500]
        )

    print(
        "\nRandom distractors:"
    )

    for cid in sample["noise_ids"]:
        print(
            f"\n[{cid}]"
        )
        print(
            get_chunk_text(cid)[:300]
        )

    print(
        "\nTotal chunks:",
        sample["total_chunks"]
    )

    print(
        "Noise ratio:",
        round(
            sample["noise_ratio"],
            3
        )
    )


# ---------------------------------------------------------
# Save the constructed experiment
# ---------------------------------------------------------

retrieval_noise_pilot.to_pickle(
    "/kaggle/working/retrieval_noise_pilot.pkl"
)

retrieval_noise_pilot.to_csv(
    "/kaggle/working/retrieval_noise_pilot.csv",
    index=False
)

print(
    "\nSaved construction files:"
)

print(
    "- /kaggle/working/retrieval_noise_pilot.pkl"
)

print(
    "- /kaggle/working/retrieval_noise_pilot.csv"
)

In [ ]:
# =========================================================
# RETRIEVAL STRATEGY × NOISE
# GENERATION PHASE
#
# 240 cases
# BM25 × Dense × Hybrid
# Noise = 0, 1, 2, 4
#
# RESUMABLE — saves after every successful case
# =========================================================

import os
import time
import pandas as pd
from groq import RateLimitError

RESULTS_PATH = "/kaggle/working/retrieval_noise_results.csv"

# ---------------------------------------------------------
# Load existing results if the notebook was interrupted
# ---------------------------------------------------------

if os.path.exists(RESULTS_PATH):

    generation_results = pd.read_csv(
        RESULTS_PATH
    )

    completed_keys = set(
        zip(
            generation_results["question_id"],
            generation_results["retrieval_method"],
            generation_results["n_noise"]
        )
    )

    print(
        f"Resuming: {len(generation_results)} cases already completed."
    )

else:

    generation_results = pd.DataFrame()

    completed_keys = set()

    print("Starting generation from scratch.")


# ---------------------------------------------------------
# Generate cases
# ---------------------------------------------------------

total_cases = len(
    retrieval_noise_pilot
)

completed = len(
    completed_keys
)

print(
    f"Total cases: {total_cases}"
)

print(
    f"Already completed: {completed}"
)

print(
    f"Remaining: {total_cases - completed}"
)


for idx, row in retrieval_noise_pilot.iterrows():

    key = (
        row["question_id"],
        row["retrieval_method"],
        int(row["n_noise"])
    )

    # Skip completed cases
    if key in completed_keys:
        continue

    context_ids = row["context_ids"]

    # CSV loads list-like values as strings only if
    # resuming, so ensure this experiment dataframe
    # still contains actual lists.
    if isinstance(context_ids, str):
        context_ids = ast.literal_eval(
            context_ids
        )

    while True:

        try:

            start_time = time.time()

            answer = generate_rag_answer(
                row["question"],
                context_ids
            )

            latency = time.time() - start_time

            result = {
                "question_id":
                    row["question_id"],

                "question":
                    row["question"],

                "reference_answer":
                    row["reference_answer"],

                "retrieval_method":
                    row["retrieval_method"],

                "n_noise":
                    int(row["n_noise"]),

                "noise_ratio":
                    float(row["noise_ratio"]),

                "retrieved_ids":
                    str(row["retrieved_ids"]),

                "noise_ids":
                    str(row["noise_ids"]),

                "context_ids":
                    str(row["context_ids"]),

                "generated_answer":
                    answer,

                "latency_seconds":
                    latency
            }

            generation_results = pd.concat(
                [
                    generation_results,
                    pd.DataFrame([result])
                ],
                ignore_index=True
            )

            # Save immediately
            generation_results.to_csv(
                RESULTS_PATH,
                index=False
            )

            completed_keys.add(key)

            completed += 1

            print(
                f"[{completed}/{total_cases}] "
                f"{row['retrieval_method']} | "
                f"noise={int(row['n_noise'])} | "
                f"{row['question_id']} | "
                f"{latency:.2f}s"
            )

            break

        except RateLimitError:

            print(
                "Rate limit reached. "
                "Waiting 10 seconds..."
            )

            time.sleep(10)

        except Exception as e:

            print(
                f"ERROR for {key}: {e}"
            )

            # Wait briefly before retrying
            time.sleep(3)


print("\n" + "=" * 70)
print("GENERATION COMPLETE")
print("=" * 70)

print(
    "Completed cases:",
    len(generation_results)
)

print(
    "Saved to:",
    RESULTS_PATH
)

In [ ]:
import os
import pandas as pd

RESULTS_PATH = "/kaggle/working/retrieval_noise_results.csv"

if os.path.exists(RESULTS_PATH):

    results = pd.read_csv(RESULTS_PATH)

    print("Saved rows:", len(results))

    print("\nBy retrieval method:")
    print(
        results["retrieval_method"]
        .value_counts()
        .sort_index()
    )

    print("\nBy noise level:")
    print(
        results["n_noise"]
        .value_counts()
        .sort_index()
    )

    print("\nMethod × noise:")
    print(
        results
        .groupby(
            ["retrieval_method", "n_noise"]
        )
        .size()
        .unstack(fill_value=0)
    )

else:
    print("Results file does not exist.")

In [ ]:
import pandas as pd

RESULTS_PATH = "/kaggle/working/retrieval_noise_results.csv"

results = pd.read_csv(RESULTS_PATH)

print("Rows:", len(results))

print("\nExact method/noise combinations:")
print(
    results
    .groupby(["retrieval_method", "n_noise"])
    .size()
    .reset_index(name="count")
    .to_string(index=False)
)

print("\nMissing values:")
print(
    results.isna().sum()
)

print("\nDuplicate experiment keys:")
keys = [
    "question_id",
    "retrieval_method",
    "n_noise"
]

duplicates = results[
    results.duplicated(
        subset=keys,
        keep=False
    )
].sort_values(keys)

print(
    duplicates[keys].to_string(index=False)
)

print(
    "\nNumber of duplicate keys:",
    duplicates[keys].drop_duplicates().shape[0]
)

print("\nUnexpected noise values:")
print(
    sorted(
        set(results["n_noise"].dropna())
        - {0, 1, 2, 4}
    )
)

In [ ]:
# ============================================================
# RESUME RETRIEVAL × NOISE EXPERIMENT
# Finite retries + checkpointing + graceful stop
# ============================================================

RESULTS_PATH = "/kaggle/working/retrieval_noise_results.csv"

# Load existing results
if os.path.exists(RESULTS_PATH):
    generation_results = pd.read_csv(RESULTS_PATH)
else:
    generation_results = pd.DataFrame()

completed_keys = set(
    zip(
        generation_results["question_id"],
        generation_results["retrieval_method"],
        generation_results["n_noise"].astype(int)
    )
)

remaining = []

for _, row in retrieval_noise_pilot.iterrows():
    key = (
        row["question_id"],
        row["retrieval_method"],
        int(row["n_noise"])
    )

    if key not in completed_keys:
        remaining.append(row)

remaining_df = pd.DataFrame(remaining)

print("Already completed:", len(completed_keys))
print("Remaining:", len(remaining_df))
print()

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

MAX_NEW_CASES = 10          # Safe first batch
MAX_RETRIES = 3              # Never retry forever
RATE_LIMIT_WAIT = 60         # seconds
BETWEEN_CALLS = 8            # seconds

successful_this_run = 0

# ------------------------------------------------------------
# Run remaining cases
# ------------------------------------------------------------

for _, row in remaining_df.iterrows():

    if successful_this_run >= MAX_NEW_CASES:
        print("\nBatch limit reached.")
        break

    key = (
        row["question_id"],
        row["retrieval_method"],
        int(row["n_noise"])
    )

    print(
        f"\nCase {successful_this_run + 1}/{MAX_NEW_CASES} | "
        f"{row['question_id']} | "
        f"{row['retrieval_method']} | "
        f"noise={int(row['n_noise'])}"
    )

    context_ids = row["context_ids"]

    if isinstance(context_ids, str):
        context_ids = ast.literal_eval(context_ids)

    success = False

    for attempt in range(1, MAX_RETRIES + 1):

        try:
            start_time = time.time()

            answer = generate_rag_answer(
                row["question"],
                context_ids
            )

            latency = time.time() - start_time

            result = {
                "question_id": row["question_id"],
                "question": row["question"],
                "retrieval_method": row["retrieval_method"],
                "n_noise": int(row["n_noise"]),
                "retrieved_ids": row["retrieved_ids"],
                "noise_ids": row["noise_ids"],
                "context_ids": row["context_ids"],
                "noise_ratio": row["noise_ratio"],
                "generated_answer": answer,
                "latency_seconds": latency
            }

            generation_results = pd.concat(
                [
                    generation_results,
                    pd.DataFrame([result])
                ],
                ignore_index=True
            )

            # Save immediately after every successful call
            generation_results.to_csv(
                RESULTS_PATH,
                index=False
            )

            completed_keys.add(key)
            successful_this_run += 1
            success = True

            print(
                f"SUCCESS | latency={latency:.1f}s | "
                f"saved={len(generation_results)}"
            )

            # Reduce TPM pressure
            time.sleep(BETWEEN_CALLS)

            break

        except RateLimitError as e:

            print(
                f"RATE LIMIT | attempt {attempt}/{MAX_RETRIES}"
            )

            if attempt < MAX_RETRIES:
                print(
                    f"Waiting {RATE_LIMIT_WAIT}s before retry..."
                )
                time.sleep(RATE_LIMIT_WAIT)

        except Exception as e:

            print(
                f"ERROR on {row['question_id']}: "
                f"{type(e).__name__}: {e}"
            )

            # Do not destroy the whole experiment because of
            # one unexpected case.
            break

    # --------------------------------------------------------
    # If this case remains blocked after retries, stop cleanly
    # --------------------------------------------------------

    if not success:

        print("\nStopping safely.")
        print("No further API calls will be made in this run.")
        print(
            f"Checkpoint saved at: {RESULTS_PATH}"
        )
        break


# ============================================================
# FINAL STATUS
# ============================================================

completed_now = pd.read_csv(RESULTS_PATH)

print("\n" + "=" * 60)
print("CURRENT EXPERIMENT STATUS")
print("=" * 60)

print("Completed:", len(completed_now), "/ 240")
print("Remaining:", 240 - len(completed_now))
print("Added this run:", successful_this_run)

print("\nDistribution:")
print(
    completed_now
    .groupby(["retrieval_method", "n_noise"])
    .size()
    .to_string()
)

print("\nSaved:", RESULTS_PATH)

In [ ]:
# ============================================================
# RETRIEVAL × NOISE — RATE-LIMIT SAFE RESUME
# One call at a time, checkpoint after every success
# ============================================================

RESULTS_PATH = "/kaggle/working/retrieval_noise_results.csv"

# Load checkpoint
generation_results = pd.read_csv(RESULTS_PATH)

completed_keys = set(
    zip(
        generation_results["question_id"],
        generation_results["retrieval_method"],
        generation_results["n_noise"].astype(int)
    )
)

# Find only unfinished cases
remaining = []

for _, row in retrieval_noise_pilot.iterrows():

    key = (
        row["question_id"],
        row["retrieval_method"],
        int(row["n_noise"])
    )

    if key not in completed_keys:
        remaining.append(row)

remaining_df = pd.DataFrame(remaining)

print(f"Completed: {len(generation_results)} / 240")
print(f"Remaining: {len(remaining_df)}")

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

MAX_NEW_CASES = 10
BETWEEN_CALLS = 15

successful = 0

# ------------------------------------------------------------
# Process cases
# ------------------------------------------------------------

for _, row in remaining_df.iterrows():

    if successful >= MAX_NEW_CASES:
        break

    question_id = row["question_id"]
    method = row["retrieval_method"]
    n_noise = int(row["n_noise"])

    print(
        f"\nCase {successful + 1}/{MAX_NEW_CASES} | "
        f"{question_id} | {method} | noise={n_noise}"
    )

    context_ids = row["context_ids"]

    if isinstance(context_ids, str):
        context_ids = ast.literal_eval(context_ids)

    try:

        start_time = time.time()

        answer = generate_rag_answer(
            row["question"],
            context_ids
        )

        latency = time.time() - start_time

        result = {
            "question_id": question_id,
            "question": row["question"],
            "retrieval_method": method,
            "n_noise": n_noise,
            "retrieved_ids": row["retrieved_ids"],
            "noise_ids": row["noise_ids"],
            "context_ids": row["context_ids"],
            "noise_ratio": row["noise_ratio"],
            "generated_answer": answer,
            "latency_seconds": latency
        }

        generation_results = pd.concat(
            [
                generation_results,
                pd.DataFrame([result])
            ],
            ignore_index=True
        )

        # IMPORTANT:
        # Save immediately after every successful generation.
        generation_results.to_csv(
            RESULTS_PATH,
            index=False
        )

        completed_keys.add(
            (question_id, method, n_noise)
        )

        successful += 1

        print(
            f"SUCCESS | "
            f"latency={latency:.2f}s | "
            f"total={len(generation_results)}/240"
        )

        # Give the API some breathing room
        time.sleep(BETWEEN_CALLS)

    except RateLimitError:

        print("\nRATE LIMIT DETECTED.")
        print("Stopping immediately instead of waiting indefinitely.")
        print("Checkpoint is already saved.")

        break

    except Exception as e:

        print(
            f"\nERROR: {type(e).__name__}: {e}"
        )
        print("Skipping this case and continuing.")

        continue


# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

generation_results = pd.read_csv(RESULTS_PATH)

print("\n" + "=" * 60)
print("UPDATED STATUS")
print("=" * 60)

print(f"Completed: {len(generation_results)} / 240")
print(f"Remaining: {240 - len(generation_results)}")
print(f"Added this run: {successful}")

print("\nDistribution:")

print(
    generation_results
    .groupby(["retrieval_method", "n_noise"])
    .size()
    .to_string()
)

print(
    f"\nCheckpoint: {RESULTS_PATH}"
)

In [ ]:
# =========================================================dont run these firsthand
# AGGREGATE RETRIEVAL × NOISE RESULTS
# =========================================================

import pandas as pd
import numpy as np

RESULT_FILE = "/kaggle/working/retrieval_noise_results.csv"

# ---------------------------------------------------------
# Load the actual completed 240-case experiment
# ---------------------------------------------------------

results = pd.read_csv(RESULT_FILE)

print("=" * 70)
print("RETRIEVAL × NOISE RESULTS")
print("=" * 70)

print(f"Loaded rows: {len(results)}")

# Safety check
assert len(results) == 240, (
    f"Expected 240 rows, found {len(results)}"
)

# ---------------------------------------------------------
# Aggregate by retrieval method × noise
# ---------------------------------------------------------

summary = (
    results
    .groupby(
        ["retrieval_method", "n_noise"]
    )
    .agg(
        mean_token_f1=("token_f1", "mean"),
        std_token_f1=("token_f1", "std"),

        mean_semantic_similarity=(
            "semantic_similarity",
            "mean"
        ),

        std_semantic_similarity=(
            "semantic_similarity",
            "std"
        ),

        mean_latency=(
            "latency_seconds",
            "mean"
        ),

        n=("question_id", "count")
    )
    .reset_index()
)

# ---------------------------------------------------------
# Mean noise ratio
# ---------------------------------------------------------

noise_ratio = (
    results
    .groupby(
        ["retrieval_method", "n_noise"]
    )["noise_ratio"]
    .mean()
    .reset_index(
        name="noise_ratio_mean"
    )
)

summary = summary.merge(
    noise_ratio,
    on=[
        "retrieval_method",
        "n_noise"
    ],
    how="left"
)

# ---------------------------------------------------------
# Baseline-relative degradation
# Baseline = same retrieval method with n_noise = 0
# ---------------------------------------------------------

baseline = (
    summary[
        summary["n_noise"] == 0
    ][
        [
            "retrieval_method",
            "mean_token_f1",
            "mean_semantic_similarity"
        ]
    ]
    .rename(
        columns={
            "mean_token_f1":
                "baseline_token_f1",

            "mean_semantic_similarity":
                "baseline_semantic_similarity"
        }
    )
)

summary = summary.merge(
    baseline,
    on="retrieval_method",
    how="left"
)

# ---------------------------------------------------------
# Absolute change from baseline
# ---------------------------------------------------------

summary["f1_change_vs_baseline"] = (
    summary["mean_token_f1"]
    - summary["baseline_token_f1"]
)

summary["semantic_change_vs_baseline"] = (
    summary["mean_semantic_similarity"]
    - summary["baseline_semantic_similarity"]
)

# ---------------------------------------------------------
# Relative percentage change
# ---------------------------------------------------------

summary["f1_relative_change_pct"] = (
    summary["f1_change_vs_baseline"]
    / summary["baseline_token_f1"]
    * 100
)

summary["semantic_relative_change_pct"] = (
    summary["semantic_change_vs_baseline"]
    / summary["baseline_semantic_similarity"]
    * 100
)

# ---------------------------------------------------------
# Display
# ---------------------------------------------------------

display(
    summary.round(4)
)

print("\n" + "=" * 70)
print("AGGREGATION COMPLETE")
print("=" * 70)

In [ ]:
import os
import glob

print("CSV files currently in /kaggle/working:")
for f in sorted(glob.glob("/kaggle/working/*.csv")):
    print(f"{os.path.basename(f):40} {os.path.getsize(f):>10,} bytes")

In [ ]:
import pandas as pd

path = "/kaggle/working/retrieval_noise_results.csv"

df = pd.read_csv(path)

print("Rows:", len(df))
print("Columns:")
print(df.columns.tolist())

display(df.head())

In [ ]:
import pandas as pd

path = "/kaggle/working/retrieval_noise_results.csv"

df = pd.read_csv(path)

print("Total rows:", len(df))
print("\nRetrieval methods:")
print(df["retrieval_method"].value_counts())

print("\nNoise levels:")
print(sorted(df["n_noise"].unique()))

print("\nNoise ratios:")
print(sorted(df["noise_ratio"].unique()))

print("\nQuestions:", df["question_id"].nunique())

print("\nMissing values:")
print(df[[
    "question_id",
    "retrieval_method",
    "n_noise",
    "noise_ratio",
    "latency_seconds",
    "token_f1",
    "semantic_similarity"
]].isna().sum())

In [ ]:
# ============================================================
# RESEARCH 1 — STATISTICAL TEST OF NOISE EFFECT
# Baseline vs increasing distractor levels
# ============================================================

import pandas as pd
import numpy as np
from scipy.stats import wilcoxon

# ------------------------------------------------------------
# Load verified Research 1 results
# ------------------------------------------------------------

results = pd.read_csv(
    "/kaggle/working/retrieval_noise_results.csv"
)

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

assert len(results) == 240, f"Expected 240 rows, found {len(results)}"
assert results["question_id"].nunique() == 20
assert set(results["retrieval_method"].unique()) == {
    "bm25", "dense", "hybrid"
}

print("Dataset verified:")
print(f"Rows: {len(results)}")
print(f"Questions: {results['question_id'].nunique()}")
print(f"Retrieval methods: {results['retrieval_method'].unique().tolist()}")
print(f"Noise levels: {sorted(results['n_noise'].unique())}")


# ------------------------------------------------------------
# Paired Wilcoxon comparisons
#
# Each noisy condition is compared against the SAME
# question + retrieval method at n_noise = 0.
# ------------------------------------------------------------

def baseline_comparisons(df, experiment_name):

    metrics = [
        "token_f1",
        "semantic_similarity",
        "latency_seconds"
    ]

    noise_levels = sorted(
        n for n in df["n_noise"].unique()
        if n != 0
    )

    rows = []

    for method in sorted(df["retrieval_method"].unique()):

        method_df = df[
            df["retrieval_method"] == method
        ].copy()

        baseline = (
            method_df[method_df["n_noise"] == 0]
            .set_index("question_id")
        )

        for noise_level in noise_levels:

            noisy = (
                method_df[
                    method_df["n_noise"] == noise_level
                ]
                .set_index("question_id")
            )

            # Ensure paired observations
            common_ids = baseline.index.intersection(
                noisy.index
            )

            for metric in metrics:

                x = baseline.loc[
                    common_ids, metric
                ].astype(float)

                y = noisy.loc[
                    common_ids, metric
                ].astype(float)

                # Wilcoxon signed-rank test
                try:
                    statistic, p_value = wilcoxon(
                        x,
                        y,
                        alternative="two-sided"
                    )
                except ValueError:
                    statistic = np.nan
                    p_value = np.nan

                baseline_mean = x.mean()
                noisy_mean = y.mean()

                absolute_change = (
                    noisy_mean - baseline_mean
                )

                if baseline_mean != 0:
                    relative_change_pct = (
                        absolute_change /
                        abs(baseline_mean)
                    ) * 100
                else:
                    relative_change_pct = np.nan

                rows.append({
                    "experiment": experiment_name,
                    "retrieval_method": method,
                    "n_noise": noise_level,
                    "metric": metric,
                    "n_pairs": len(common_ids),
                    "baseline_mean": baseline_mean,
                    "noise_mean": noisy_mean,
                    "absolute_change": absolute_change,
                    "relative_change_pct": relative_change_pct,
                    "wilcoxon_statistic": statistic,
                    "p_value": p_value,
                    "significant_p05": (
                        p_value < 0.05
                        if not np.isnan(p_value)
                        else False
                    )
                })

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# Run analysis
# ------------------------------------------------------------

stats_results = baseline_comparisons(
    results,
    "Research 1 — Appended Distractor Context Effects"
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

display(stats_results)

In [ ]:
# ============================================================
# RESEARCH 1 — DESCRIPTIVE PERFORMANCE BY NOISE LEVEL
# ============================================================

import pandas as pd
import numpy as np

results = pd.read_csv(
    "/kaggle/working/retrieval_noise_results.csv"
)

# Aggregate by retrieval method and noise level
descriptive = (
    results
    .groupby(["retrieval_method", "n_noise", "noise_ratio"])
    .agg(
        token_f1_mean=("token_f1", "mean"),
        token_f1_std=("token_f1", "std"),

        semantic_similarity_mean=("semantic_similarity", "mean"),
        semantic_similarity_std=("semantic_similarity", "std"),

        latency_mean=("latency_seconds", "mean"),
        latency_std=("latency_seconds", "std"),

        n=("question_id", "count")
    )
    .reset_index()
)

# Calculate percentage change relative to each method's
# zero-noise baseline
for method in descriptive["retrieval_method"].unique():

    baseline = descriptive[
        (descriptive["retrieval_method"] == method) &
        (descriptive["n_noise"] == 0)
    ].iloc[0]

    mask = descriptive["retrieval_method"] == method

    descriptive.loc[mask, "token_f1_change_pct"] = (
        (descriptive.loc[mask, "token_f1_mean"]
         - baseline["token_f1_mean"])
        / abs(baseline["token_f1_mean"])
        * 100
    )

    descriptive.loc[mask, "semantic_similarity_change_pct"] = (
        (descriptive.loc[mask, "semantic_similarity_mean"]
         - baseline["semantic_similarity_mean"])
        / abs(baseline["semantic_similarity_mean"])
        * 100
    )

    descriptive.loc[mask, "latency_change_pct"] = (
        (descriptive.loc[mask, "latency_mean"]
         - baseline["latency_mean"])
        / abs(baseline["latency_mean"])
        * 100
    )

# Round for readability
display(
    descriptive.round({
        "noise_ratio": 4,
        "token_f1_mean": 4,
        "token_f1_std": 4,
        "semantic_similarity_mean": 4,
        "semantic_similarity_std": 4,
        "latency_mean": 4,
        "latency_std": 4,
        "token_f1_change_pct": 2,
        "semantic_similarity_change_pct": 2,
        "latency_change_pct": 2
    })
)

In [ ]:
# ============================================================
# RESEARCH 1 — MULTIPLE COMPARISON CORRECTION
# Holm-Bonferroni correction across all 27 tests
# ============================================================

import pandas as pd
import numpy as np
from scipy.stats import false_discovery_control

# ------------------------------------------------------------
# Load the statistical results produced previously
# ------------------------------------------------------------

# If stats_results is still in memory, use it.
# Otherwise recreate it from the saved CSV.

if "stats_results" not in globals():
    raise RuntimeError(
        "stats_results is not available. "
        "Please rerun the previous Wilcoxon analysis cell first."
    )

corrected = stats_results.copy()

# ------------------------------------------------------------
# Holm correction
# ------------------------------------------------------------

p_values = corrected["p_value"].to_numpy()

# scipy's false_discovery_control supports BH/BY,
# so implement Holm-Bonferroni explicitly.

valid_mask = ~np.isnan(p_values)

valid_p = p_values[valid_mask]

order = np.argsort(valid_p)
sorted_p = valid_p[order]

m = len(sorted_p)

holm_sorted = np.empty(m)

for i, p in enumerate(sorted_p):
    holm_sorted[i] = min(
        1.0,
        (m - i) * p
    )

# Enforce monotonicity
for i in range(m - 2, -1, -1):
    holm_sorted[i] = min(
        holm_sorted[i],
        holm_sorted[i + 1]
    )

# Put corrected values back into original order
holm_values = np.empty(m)
holm_values[order] = holm_sorted

corrected.loc[valid_mask, "holm_p_value"] = holm_values

# ------------------------------------------------------------
# Corrected significance
# ------------------------------------------------------------

corrected["significant_holm_p05"] = (
    corrected["holm_p_value"] < 0.05
)

# ------------------------------------------------------------
# Sort by corrected p-value
# ------------------------------------------------------------

corrected = corrected.sort_values(
    "holm_p_value",
    na_position="last"
).reset_index(drop=True)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display(
    corrected[
        [
            "retrieval_method",
            "n_noise",
            "metric",
            "n_pairs",
            "baseline_mean",
            "noise_mean",
            "relative_change_pct",
            "p_value",
            "holm_p_value",
            "significant_holm_p05"
        ]
    ].round({
        "baseline_mean": 4,
        "noise_mean": 4,
        "relative_change_pct": 2,
        "p_value": 4,
        "holm_p_value": 4
    })
)

print("\nTotal tests:", len(corrected))
print(
    "Raw p < 0.05:",
    (corrected["p_value"] < 0.05).sum()
)
print(
    "Holm-adjusted p < 0.05:",
    corrected["significant_holm_p05"].sum()
)

In [ ]:
# ============================================================
# RESEARCH 1 — CONSTRUCTION-DEPENDENT INTRUSION DIAGNOSTIC
# ============================================================

import pandas as pd
import ast
import numpy as np

results = pd.read_csv(
    "/kaggle/working/retrieval_noise_results.csv"
)

# ------------------------------------------------------------
# Safely convert stored list strings back into Python lists
# ------------------------------------------------------------

def parse_list(value):
    if pd.isna(value):
        return []
    
    if isinstance(value, list):
        return value
    
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []


results["retrieved_ids_parsed"] = (
    results["retrieved_ids"]
    .apply(parse_list)
)

results["noise_ids_parsed"] = (
    results["noise_ids"]
    .apply(parse_list)
)

results["context_ids_parsed"] = (
    results["context_ids"]
    .apply(parse_list)
)

# ------------------------------------------------------------
# Diagnostic only: distractors are appended after retrieval, so intrusion into retrieved_ids is guaranteed to be zero. This is not a retriever robustness test.
#
# contamination = retrieved noise documents / retrieved docs
# ------------------------------------------------------------

def contamination(row):

    retrieved = set(row["retrieved_ids_parsed"])
    noise = set(row["noise_ids_parsed"])

    if len(retrieved) == 0:
        return 0.0

    return len(retrieved & noise) / len(retrieved)


def noise_retrieved_count(row):

    retrieved = set(row["retrieved_ids_parsed"])
    noise = set(row["noise_ids_parsed"])

    return len(retrieved & noise)


results["noise_retrieved_count"] = (
    results.apply(noise_retrieved_count, axis=1)
)

results["retrieval_contamination"] = (
    results.apply(contamination, axis=1)
)

results["retrieved_count"] = (
    results["retrieved_ids_parsed"]
    .apply(len)
)

# ------------------------------------------------------------
# Aggregate by retrieval method and noise level
# ------------------------------------------------------------

contamination_summary = (
    results
    .groupby(
        ["retrieval_method", "n_noise", "noise_ratio"]
    )
    .agg(
        mean_contamination=(
            "retrieval_contamination",
            "mean"
        ),
        std_contamination=(
            "retrieval_contamination",
            "std"
        ),
        mean_noise_retrieved=(
            "noise_retrieved_count",
            "mean"
        ),
        mean_retrieved=(
            "retrieved_count",
            "mean"
        ),
        n=("question_id", "count")
    )
    .reset_index()
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display(
    contamination_summary.round({
        "noise_ratio": 4,
        "mean_contamination": 4,
        "std_contamination": 4,
        "mean_noise_retrieved": 4,
        "mean_retrieved": 4
    })
)

In [ ]:
# ============================================================
# RESEARCH 1 — CONSTRUCTION-DEPENDENT TOP-5 PRESERVATION DIAGNOSTIC
# ============================================================

import pandas as pd
import ast

results = pd.read_csv(
    "/kaggle/working/retrieval_noise_results.csv"
)

def parse_list(value):
    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []


results["retrieved_ids_parsed"] = (
    results["retrieved_ids"].apply(parse_list)
)

results["context_ids_parsed"] = (
    results["context_ids"].apply(parse_list)
)

# ------------------------------------------------------------
# Diagnostic only: the top-five is selected before distractors are appended; preservation is guaranteed by construction.
# ------------------------------------------------------------

def context_recall(row):

    retrieved = set(row["retrieved_ids_parsed"])
    context = set(row["context_ids_parsed"])

    if len(context) == 0:
        return 0.0

    return len(retrieved & context) / len(context)


def context_overlap_count(row):

    retrieved = set(row["retrieved_ids_parsed"])
    context = set(row["context_ids_parsed"])

    return len(retrieved & context)


results["context_recall"] = (
    results.apply(context_recall, axis=1)
)

results["context_overlap_count"] = (
    results.apply(context_overlap_count, axis=1)
)

# ------------------------------------------------------------
# Aggregate
# ------------------------------------------------------------

context_summary = (
    results
    .groupby(
        ["retrieval_method", "n_noise", "noise_ratio"]
    )
    .agg(
        mean_context_recall=(
            "context_recall",
            "mean"
        ),
        std_context_recall=(
            "context_recall",
            "std"
        ),
        mean_context_overlap=(
            "context_overlap_count",
            "mean"
        ),
        n=("question_id", "count")
    )
    .reset_index()
)

display(
    context_summary.round({
        "noise_ratio": 4,
        "mean_context_recall": 4,
        "std_context_recall": 4,
        "mean_context_overlap": 4
    })
)

In [ ]:
# ============================================================
# DEBUG — INSPECT RETRIEVED IDS VS CONTEXT IDS
# ============================================================

import pandas as pd
import ast

results = pd.read_csv(
    "/kaggle/working/retrieval_noise_results.csv"
)

def parse_list(value):
    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []

for i in range(5):
    row = results.iloc[i]

    retrieved = parse_list(row["retrieved_ids"])
    context = parse_list(row["context_ids"])

    print("=" * 70)
    print("Question:", row["question_id"])
    print("Method:", row["retrieval_method"])
    print("Noise:", row["n_noise"])

    print("\nRetrieved IDs:")
    print(retrieved)

    print("\nContext IDs:")
    print(context)

    print("\nRetrieved count:", len(retrieved))
    print("Context count:", len(context))

    print(
        "Intersection:",
        set(retrieved) & set(context)
    )

In [ ]:
# ============================================================
# CONSTRUCTION-DEPENDENT BASELINE TOP-5 PRESERVATION
# Diagnostic only: retrieval is not rerun after distractors are appended.
# This cannot measure whether distractors displace documents during retrieval.
# ============================================================

import pandas as pd
import ast
import numpy as np

results = pd.read_csv(
    "/kaggle/working/retrieval_noise_results.csv"
)


def parse_list(value):
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []


# Parse document ID columns
results["retrieved_list"] = results["retrieved_ids"].apply(parse_list)
results["context_list"] = results["context_ids"].apply(parse_list)
results["noise_list"] = results["noise_ids"].apply(parse_list)


# ------------------------------------------------------------
# Build the noise-free baseline document set
# One baseline set for each question × retrieval method
# ------------------------------------------------------------

baseline = (
    results[results["n_noise"] == 0]
    .set_index(["question_id", "retrieval_method"])
    ["retrieved_list"]
    .to_dict()
)


def calculate_preservation(row):
    key = (row["question_id"], row["retrieval_method"])

    baseline_docs = set(baseline.get(key, []))
    retrieved_docs = set(row["retrieved_list"])

    if not baseline_docs:
        return np.nan

    return len(baseline_docs & retrieved_docs) / len(baseline_docs)


def calculate_overlap(row):
    key = (row["question_id"], row["retrieval_method"])

    baseline_docs = set(baseline.get(key, []))
    retrieved_docs = set(row["retrieved_list"])

    return len(baseline_docs & retrieved_docs)


results["baseline_preservation"] = results.apply(
    calculate_preservation,
    axis=1
)

results["baseline_overlap"] = results.apply(
    calculate_overlap,
    axis=1
)


# ------------------------------------------------------------
# Aggregate noisy conditions
# ------------------------------------------------------------

preservation_summary = (
    results
    .groupby(
        ["retrieval_method", "n_noise", "noise_ratio"],
        as_index=False
    )
    .agg(
        mean_baseline_preservation=(
            "baseline_preservation",
            "mean"
        ),
        std_baseline_preservation=(
            "baseline_preservation",
            "std"
        ),
        mean_baseline_overlap=(
            "baseline_overlap",
            "mean"
        ),
        n=("question_id", "count")
    )
)


print("\nBASELINE CONTEXT PRESERVATION")
print("=" * 80)

print(
    preservation_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

## Baseline Top-5 Preservation — Construction CaveatThe experiment-construction cell retrieves the top five documents from the original corpus first, then appends distractor IDs to the context passed to the answer generator. Therefore, the baseline top-five documents are preserved by construction. This metric does **not** test whether distractors added to the retrieval candidate corpus would displace relevant results.A genuine retrieval-robustness experiment must inject distractor documents into the searchable candidate corpus before retrieval and then rerun the retriever.

In [ ]:
# Construction-dependent top-5 accounting only.# Distractors are appended after retrieval, so these values do not test# whether retrievers resist distractors in the searchable candidate corpus.print("\nTOP-5 PRESERVATION / CONTAMINATION (CONSTRUCTION-DEPENDENT)")print("Interpret these as construction checks only, not retrieval-robustness evidence.")

# Research 1 — Findings and Interpretation

## Study scope

The frozen 240-row experiment evaluates distractor chunks **appended after** BM25, dense, or hybrid retrieval has already selected its original top five. It therefore measures downstream answer quality and generation-call latency under appended context; it does not test retrieval-ranking robustness.

## Construction diagnostic

The original top-five set is selected before distractors are appended. Zero distractor intrusion into the original retrieved top five and 100% baseline preservation are therefore guaranteed by construction. These values are diagnostics of the data-construction sequence, not empirical evidence that the retrievers rejected or resisted distractors.

## Answer quality

Token F1 and semantic similarity vary descriptively across conditions. None of the 18 answer-quality comparisons remained significant after Holm correction across the full 27-test family. This is not proof of equivalence or absence of any effect.

## Generation-call latency

The timer surrounds the `generate_rag_answer` call, not isolated retrieval. The only comparison reported as significant after Holm correction is the dense-method pipeline at four appended distractors: mean latency changes from 0.405087 s to 0.595432 s (+46.99%; raw p = 0.001690; Holm-adjusted p = 0.045628; paired rank-biserial correlation = 0.761905; n = 20). The result is close to the 0.05 threshold and may reflect context length, model-service variability, or other runtime effects.

## Interpretation boundary

Do not describe these findings as evidence of retrieval invariance or retriever robustness. Testing ranking robustness requires adding distractors to the searchable candidate corpus **before** retrieval and rerunning each retriever in a separately versioned experiment.

In [ ]:
# ============================================================
# RESEARCH 1 — FINAL STATISTICAL ANALYSIS
# Current verified dataset: 240 observations
# ============================================================

import pandas as pd
import numpy as np
from scipy.stats import wilcoxon

RESULTS_PATH = "/kaggle/working/retrieval_noise_results.csv"

df = pd.read_csv(RESULTS_PATH)

# ------------------------------------------------------------
# Verify dataset
# ------------------------------------------------------------

print("Rows:", len(df))
print("Questions:", df["question_id"].nunique())
print("Methods:", df["retrieval_method"].unique())
print("Noise levels:", sorted(df["n_noise"].unique()))

assert len(df) == 240, "Expected 240 observations."
assert df["question_id"].nunique() == 20
assert df["retrieval_method"].nunique() == 3


# ------------------------------------------------------------
# Paired baseline comparisons
# ------------------------------------------------------------

metrics = [
    "token_f1",
    "semantic_similarity",
    "latency_seconds"
]

statistical_results = []

for method in sorted(df["retrieval_method"].unique()):

    method_df = df[
        df["retrieval_method"] == method
    ]

    baseline = (
        method_df[method_df["n_noise"] == 0]
        .set_index("question_id")
    )

    for noise_level in [1, 2, 4]:

        noisy = (
            method_df[method_df["n_noise"] == noise_level]
            .set_index("question_id")
        )

        common_ids = baseline.index.intersection(
            noisy.index
        )

        for metric in metrics:

            baseline_values = baseline.loc[
                common_ids, metric
            ]

            noisy_values = noisy.loc[
                common_ids, metric
            ]

            stat, p_value = wilcoxon(
                baseline_values,
                noisy_values,
                alternative="two-sided"
            )

            baseline_mean = baseline_values.mean()
            noisy_mean = noisy_values.mean()

            absolute_change = (
                noisy_mean - baseline_mean
            )

            relative_change = (
                absolute_change
                / baseline_mean
                * 100
                if baseline_mean != 0
                else np.nan
            )

            statistical_results.append({

                "retrieval_method": method,

                "n_noise": noise_level,

                "noise_ratio":
                    noisy["noise_ratio"].iloc[0],

                "metric": metric,

                "baseline_mean":
                    baseline_mean,

                "noisy_mean":
                    noisy_mean,

                "absolute_change":
                    absolute_change,

                "relative_change_pct":
                    relative_change,

                "wilcoxon_stat":
                    stat,

                "p_value":
                    p_value,

                "n":
                    len(common_ids)
            })


stats_df = pd.DataFrame(statistical_results)


# ------------------------------------------------------------
# Holm correction across ALL 27 comparisons
#
# 3 methods × 3 noise levels × 3 metrics = 27 tests
# ------------------------------------------------------------

def holm_adjust(p_values):

    p_values = np.asarray(p_values)

    order = np.argsort(p_values)

    adjusted = np.empty(len(p_values))

    for rank, index in enumerate(order):

        adjusted[index] = min(
            (len(p_values) - rank) * p_values[index],
            1.0
        )

    # Ensure monotonicity
    for i in range(1, len(order)):

        previous = order[i - 1]
        current = order[i]

        adjusted[current] = max(
            adjusted[current],
            adjusted[previous]
        )

    return adjusted


stats_df["holm_p_value"] = holm_adjust(
    stats_df["p_value"].values
)

stats_df["significant_after_holm"] = (
    stats_df["holm_p_value"] < 0.05
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\n")
print("=" * 110)
print("RESEARCH 1 — PAIRED STATISTICAL ANALYSIS")
print("=" * 110)

display(
    stats_df.round(4)
)


# ------------------------------------------------------------
# Significant results only
# ------------------------------------------------------------

print("\n")
print("=" * 110)
print("SIGNIFICANT RESULTS AFTER HOLM CORRECTION")
print("=" * 110)

significant = stats_df[
    stats_df["significant_after_holm"]
]

if len(significant) == 0:

    print("No statistically significant results.")

else:

    display(
        significant.round(4)
    )


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

stats_path = (
    "/kaggle/working/"
    "research1_statistical_analysis.csv"
)

stats_df.to_csv(
    stats_path,
    index=False
)

print("\nSaved:")
print(stats_path)

# Research 1 — Statistical Analysis

The inferential family contains 27 matched paired comparisons: three retrieval methods × three non-zero appended-distractor conditions (1, 2, 4) × three metrics (Token F1, semantic similarity, generation-call latency).

Each noisy condition is compared with the same question and method at zero appended distractors using a paired Wilcoxon signed-rank test. Holm correction is applied jointly across all 27 p-values. The saved timer measures the `generate_rag_answer` call, not isolated retrieval latency.

For the canonical analysis that verifies the frozen dataset SHA-256 and recomputes all 27 tests without making LLM/API calls, use `scripts/recompute_primary_statistics.py`. Do not rerun the LLM evaluation cells merely to reproduce the statistical table.

In [ ]:
# ============================================================
# RESEARCH 1 — EFFECT SIZE
# Rank-biserial correlation for paired Wilcoxon tests
# ============================================================

def rank_biserial_from_differences(baseline, noisy):

    differences = noisy.values - baseline.values

    differences = differences[
        differences != 0
    ]

    if len(differences) == 0:
        return 0.0

    abs_diff = np.abs(differences)

    ranks = pd.Series(abs_diff).rank(
        method="average"
    ).values

    positive_rank_sum = ranks[
        differences > 0
    ].sum()

    negative_rank_sum = ranks[
        differences < 0
    ].sum()

    total_rank_sum = (
        positive_rank_sum +
        negative_rank_sum
    )

    if total_rank_sum == 0:
        return 0.0

    return (
        positive_rank_sum -
        negative_rank_sum
    ) / total_rank_sum


effect_sizes = []

for method in sorted(df["retrieval_method"].unique()):

    method_df = df[
        df["retrieval_method"] == method
    ]

    baseline = (
        method_df[method_df["n_noise"] == 0]
        .set_index("question_id")
    )

    for noise_level in [1, 2, 4]:

        noisy = (
            method_df[
                method_df["n_noise"] == noise_level
            ]
            .set_index("question_id")
        )

        common_ids = baseline.index.intersection(
            noisy.index
        )

        for metric in metrics:

            baseline_values = baseline.loc[
                common_ids, metric
            ]

            noisy_values = noisy.loc[
                common_ids, metric
            ]

            effect = rank_biserial_from_differences(
                baseline_values,
                noisy_values
            )

            effect_sizes.append({

                "retrieval_method": method,

                "n_noise": noise_level,

                "metric": metric,

                "rank_biserial_correlation":
                    effect
            })


effect_df = pd.DataFrame(effect_sizes)

stats_df = stats_df.merge(
    effect_df,
    on=[
        "retrieval_method",
        "n_noise",
        "metric"
    ],
    how="left"
)


print("=" * 100)
print("RESEARCH 1 — EFFECT SIZES")
print("=" * 100)

display(
    stats_df.round(4)
)


stats_df.to_csv(
    "/kaggle/working/research1_statistical_analysis.csv",
    index=False
)

print(
    "\nUpdated:",
    "/kaggle/working/research1_statistical_analysis.csv"
)

# Research 1 — Effect Size Analysis

Statistical significance was supplemented with rank-biserial correlation as an effect-size measure for the paired Wilcoxon comparisons.

Effect sizes are reported alongside the observed percentage changes and Holm-adjusted p-values to distinguish statistical evidence from the magnitude and direction of the observed differences.

This is particularly important because a statistically significant result does not necessarily imply a practically large effect, while a non-significant result does not establish that two conditions are identical.

In [ ]:
# ============================================================
# RESEARCH 1 — PUBLICATION-QUALITY FIGURES
# ============================================================

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Figure 1 — Construction-dependent intrusion diagnostic
# ------------------------------------------------------------

contamination = (
    results
    .groupby(["retrieval_method", "n_noise"], as_index=False)
    .agg(
        contamination_rate=(
            "noise_list",
            lambda x: np.mean([
                len(set(noise) & set(retrieved)) /
                len(retrieved)
                if len(retrieved) > 0 else 0
                for noise, retrieved in zip(
                    x,
                    results.loc[x.index, "retrieved_list"]
                )
            ])
        )
    )
)

plt.figure(figsize=(8, 5))

for method in sorted(
    contamination["retrieval_method"].unique()
):
    subset = contamination[
        contamination["retrieval_method"] == method
    ]

    plt.plot(
        subset["n_noise"],
        subset["contamination_rate"],
        marker="o",
        label=method.upper()
    )

plt.xlabel("Number of Appended Distractors")
plt.ylabel("Construction-dependent Intrusion Rate")
plt.title("Zero Intrusion by Construction (Not a Retrieval Outcome)")
plt.xticks([0, 1, 2, 4])
plt.ylim(-0.02, 1.0)
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    "/kaggle/working/research1_fig1_contamination.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# Figure 2 — Construction-dependent top-five preservation diagnostic
# ------------------------------------------------------------

preservation = (
    results
    .groupby(["retrieval_method", "n_noise"], as_index=False)
    ["baseline_preservation"]
    .mean()
)

plt.figure(figsize=(8, 5))

for method in sorted(
    preservation["retrieval_method"].unique()
):
    subset = preservation[
        preservation["retrieval_method"] == method
    ]

    plt.plot(
        subset["n_noise"],
        subset["baseline_preservation"],
        marker="o",
        label=method.upper()
    )

plt.xlabel("Number of Injected Distractors")
plt.ylabel("Construction-dependent Top-5 Preservation")
plt.title("Retrieval Stability Under Increasing Noise")
plt.xticks([0, 1, 2, 4])
plt.ylim(0.0, 1.05)
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    "/kaggle/working/research1_fig2_preservation.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# Figure 3 — Answer quality
# ------------------------------------------------------------

quality = (
    results
    .groupby(
        ["retrieval_method", "n_noise"],
        as_index=False
    )
    .agg(
        token_f1=("token_f1", "mean"),
        semantic_similarity=(
            "semantic_similarity",
            "mean"
        )
    )
)

plt.figure(figsize=(8, 5))

for method in sorted(
    quality["retrieval_method"].unique()
):

    subset = quality[
        quality["retrieval_method"] == method
    ]

    plt.plot(
        subset["n_noise"],
        subset["token_f1"],
        marker="o",
        label=f"{method.upper()} — Token F1"
    )

    plt.plot(
        subset["n_noise"],
        subset["semantic_similarity"],
        marker="s",
        linestyle="--",
        label=f"{method.upper()} — Semantic Similarity"
    )

plt.xlabel("Number of Injected Distractors")
plt.ylabel("Mean Score")
plt.title("Answer Quality Under Increasing Noise")
plt.xticks([0, 1, 2, 4])
plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    "/kaggle/working/research1_fig3_answer_quality.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# Figure 4 — Latency
# ------------------------------------------------------------

latency = (
    results
    .groupby(
        ["retrieval_method", "n_noise"],
        as_index=False
    )
    ["latency_seconds"]
    .mean()
)

plt.figure(figsize=(8, 5))

for method in sorted(
    latency["retrieval_method"].unique()
):

    subset = latency[
        latency["retrieval_method"] == method
    ]

    plt.plot(
        subset["n_noise"],
        subset["latency_seconds"],
        marker="o",
        label=method.upper()
    )

plt.xlabel("Number of Injected Distractors")
plt.ylabel("Mean Latency (seconds)")
plt.title("Retrieval Latency Under Increasing Noise")
plt.xticks([0, 1, 2, 4])
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    "/kaggle/working/research1_fig4_latency.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


print("\nSaved figures:")
print("/kaggle/working/research1_fig1_contamination.png")
print("/kaggle/working/research1_fig2_preservation.png")
print("/kaggle/working/research1_fig3_answer_quality.png")
print("/kaggle/working/research1_fig4_latency.png")

In [ ]:
import pandas as pd

RESULTS_PATH = "/kaggle/working/retrieval_noise_results.csv"

results = pd.read_csv(RESULTS_PATH)

print("Loaded:", results.shape)
print(results.columns.tolist())

In [ ]:
import os

print("Files currently in /kaggle/working:")
for root, dirs, files in os.walk("/kaggle/working"):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
import os

print("\nFiles in /kaggle/input:")
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
print("Number of executed input cells:", len(In))

for i, code in enumerate(In):
    if any(term in code.lower() for term in [
        "retrieval_noise_results",
        "baseline_preservation",
        "contamination",
        "wilcoxon",
        "research1_fig"
    ]):
        print("\n" + "=" * 80)
        print("INPUT CELL", i)
        print("=" * 80)
        print(code[:3000])

In [ ]:
import os

print("Kaggle working directory:")
print(os.listdir("/kaggle/working"))

print("\nAll files under /kaggle/working:")
for root, dirs, files in os.walk("/kaggle/working"):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
items = list(globals().items())

print("DataFrame-like variables:")

for name, obj in items:
    try:
        if isinstance(obj, pd.DataFrame):
            print(f"{name}: shape={obj.shape}")
            print("columns:", list(obj.columns))
    except Exception:
        pass

In [ ]:
import re

for i, code in enumerate(In):
    print("\n" + "=" * 100)
    print(f"CELL {i}")
    print("=" * 100)
    print(code[:1500])

## VisualizationThe figures summarize context contamination accounting, the construction-dependent top-five preservation metric, answer-quality scores, and measured generation-call latency.**Interpretation caveat:** because distractors are appended after the initial top-five retrieval, the current notebook does not measure retriever ranking robustness. The latency timer surrounds the answer-generation function call and is not an isolated retrieval-latency measurement.

In [ ]:
# ============================================================
# RESEARCH 1 — MAIN RESULTS TABLE
# ============================================================

main_table = (
    results
    .groupby(
        ["retrieval_method", "n_noise", "noise_ratio"],
        as_index=False
    )
    .agg(
        token_f1_mean=("token_f1", "mean"),
        token_f1_std=("token_f1", "std"),

        semantic_similarity_mean=(
            "semantic_similarity",
            "mean"
        ),

        semantic_similarity_std=(
            "semantic_similarity",
            "std"
        ),

        latency_mean=(
            "latency_seconds",
            "mean"
        ),

        latency_std=(
            "latency_seconds",
            "std"
        ),

        baseline_preservation=(
            "baseline_preservation",
            "mean"
        ),

        n=("question_id", "count")
    )
)

print("=" * 110)
print("RESEARCH 1 — MAIN RESULTS TABLE")
print("=" * 110)

display(
    main_table.round(4)
)

main_table.to_csv(
    "/kaggle/working/research1_main_results.csv",
    index=False
)

print(
    "\nSaved:",
    "/kaggle/working/research1_main_results.csv"
)

# Research 1 — Main Results

The main results table reports descriptive performance for every retrieval method and noise condition.

Reported measures include:

- Mean and standard deviation of token-level F1
- Mean and standard deviation of semantic similarity
- Mean and standard deviation of latency
- Mean baseline top-5 preservation
- Number of observations

Inferential conclusions are based on the paired Wilcoxon analysis with Holm correction rather than descriptive mean differences alone.

In [ ]:
# Final evidence summary — do not rerun the LLM evaluation here.
# The older summary cell combined construction diagnostics with retrieval-robustness
# claims and depended on stale intermediate variables. The primary analysis is now
# reproducible from the frozen CSV using scripts/recompute_primary_statistics.py.
print("Study scope: appended-context distractors; retrieval is not rerun.")
print("Top-five preservation / zero intrusion are construction diagnostics only.")
print("Latency is measured around generate_rag_answer, not isolated retrieval.")
print("See scripts/recompute_primary_statistics.py for the canonical 27-test analysis.")

# Research 1 — Final Evidence Summary

The frozen primary experiment contains 240 observations: 20 questions × 3 retrieval methods × 4 appended-context distractor levels (0, 1, 2, 4).

### What was evaluated

- Token F1 and semantic similarity of generated answers.
- Generation-call latency, timed around `generate_rag_answer`.
- 27 paired comparisons with Holm correction.

### What was not evaluated

Retrieval was not rerun after distractors were added. Therefore, zero intrusion and 100% preservation of the original top five are guaranteed by construction and cannot establish retrieval-ranking robustness.

### Statistical interpretation

No answer-quality comparison remained significant after Holm correction. One dense-method generation-call latency contrast at four appended distractors remained significant (adjusted p = 0.045628; n = 20). This is a narrow, implementation-specific finding that needs replication and should not be attributed to retrieval latency.

# Research 1 — Final Experimental Summary## ObjectiveThe frozen experiment evaluates how appended distractor chunks in the generation context relate to answer quality and generation-call latency after BM25, dense, or hybrid retrieval.## Experimental DesignThe experiment contains **240 observations** across 20 questions, 3 retrieval methods, and 4 appended-context distractor levels (0, 1, 2, 4). Top-k retrieval is performed first; distractors are then appended to the context.## Statistical AnalysisFor each method, each non-zero noise condition is compared with the matched zero-noise condition for Token F1, semantic similarity, and generation-call latency. This yields **27 paired comparisons** (3 methods × 3 non-zero contrasts × 3 metrics), with Holm correction across the complete family.## Interpretation BoundaryTop-five preservation and zero intrusion are guaranteed by the construction and must not be interpreted as empirical evidence of retrieval robustness. The timer surrounds the answer-generation call, so latency includes downstream generation/runtime effects and is not isolated retrieval latency. A separate candidate-pool perturbation experiment is required to test retrieval-ranking robustness.

# Research 1 — Conclusion

This experiment evaluated appended distractor context, not retrieval robustness. Across 240 observations, distractor chunks were added only after the original top-five had been retrieved; consequently, zero intrusion and 100% baseline top-five preservation are construction artifacts and cannot establish ranking stability.

No Token F1 or semantic-similarity comparison remained significant after Holm correction across the 27 planned paired tests. The only comparison that remained significant was generation-call latency for the dense-method pipeline at four appended distractors: 0.405087 s at baseline versus 0.595432 s with four distractors (+46.99%; Holm-adjusted p = 0.045628; paired rank-biserial correlation = 0.761905; n = 20). The timer surrounds the answer-generation call and must not be described as isolated retrieval latency.

A separate experiment that inserts distractors into the searchable candidate corpus before rerunning each retriever is required to answer the original retrieval-ranking robustness question.